<a href="https://colab.research.google.com/github/Tecknique/200_ml/blob/main/FLA_UW_GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# /content/uwlfa_gui_cells/cell1_config.py

import io
import json
import re
import shutil
import threading
import zipfile
import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

from IPython.display import HTML, display

# ----------------- config -----------------
OWNER, REPO = "timrobinson", "UW-LFA-Analysis"
BRANCH = "main"
BASE_REL = Path("100microliters/Database")
IMG_EXTS = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

EXPORT_ROOT = Path("/content/roi_exports") if Path("/content").exists() else Path.cwd() / "roi_exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)


In [2]:
import shutil
from pathlib import Path

# Define the root export directory and the specific date folder
EXPORT_ROOT = Path("/content/roi_exports")
DATE_FOLDER = "2026-01-26"  # # /content/uwlfa_gui_cells/cell2_deps.py

try:
    import cv2
    import numpy as np
    import requests
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.signal import savgol_filter
except Exception:
    import sys
    import subprocess as sp

    sp.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opencv-python-headless",
            "requests",
            "flask",
            "matplotlib",
            "pandas",
            "scipy",
        ],
        check=True,
    )
    import cv2  # noqa
    import numpy as np  # noqa
    import requests  # noqa
    import pandas as pd  # noqa
    import matplotlib  # noqa

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # noqa
    from scipy.signal import savgol_filter  # noqa
# This should dynamically match the DATE_FOLDER from the first cell

folder_to_delete = EXPORT_ROOT / DATE_FOLDER

if folder_to_delete.exists():
    print(f"Deleting folder: {folder_to_delete.as_posix()}")
    shutil.rmtree(folder_to_delete)
    print("Folder deleted successfully.")
else:
    print(f"Folder not found: {folder_to_delete.as_posix()}")


Folder not found: /content/roi_exports/2026-01-26


In [3]:
# /content/uwlfa_gui_cells/cell3_helpers.py

def _download_repo_zip(owner: str, repo: str, branch: str) -> bytes:
    def fetch(br: str) -> Optional[bytes]:
        url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{br}.zip"
        r = requests.get(url, timeout=60)
        return r.content if r.status_code == 200 else None

    blob = fetch(branch) or (fetch("master") if branch == "main" else None)
    if blob is None:
        raise RuntimeError(f"Cannot download {owner}/{repo} ({branch}/master).")
    return blob


def _extract_zip_to_tmp(zip_bytes: bytes, base_dir: Path) -> Path:
    tmp_root = base_dir / "_uwlfa_tmp"
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    tmp_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as zf:
        zf.extractall(tmp_root)
        top_dirs = sorted({Path(n).parts[0] for n in zf.namelist() if "/" in n})

    if not top_dirs:
        raise RuntimeError("Unexpected zip structure.")
    return tmp_root / top_dirs[0]


def sanitize_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", (s or "").strip())


def row_name_from_filename(filename: str) -> str:
    stem = Path(filename).stem
    if "__" in stem:
        return stem.split("__", 1)[0]
    return stem.split("_", 1)[0] if "_" in stem else stem


def row_sort_key(row: str) -> Tuple[int, float, str]:
    r = (row or "").strip()
    loads_order = {
        "neg": (0, -1.0),
        "1e5": (1, 1e5),
        "1.5e5": (2, 1.5e5),
        "5e5": (3, 5e5),
        "1e6": (4, 1e6),
        "5e6": (5, 5e6),
        "1e7": (6, 1e7),
        "cc": (7, 7e7),
        "k": (8, 8e7),
        "dip": (9, 9e7),
        "pos": (10, 1e12),
    }
    key = loads_order.get(r.lower())
    if key:
        return (0, key[0], r.lower())
    m = re.match(r"^(\d+(?:\.\d+)?)e(\d+)$", r.lower())
    if m:
        base = float(m.group(1))
        exp = float(m.group(2))
        return (1, base * (10**exp), r.lower())
    return (2, float("inf"), r.lower())


def _png_message(msg: str) -> bytes:
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.text(0.02, 0.6, msg, ha="left", va="center", fontsize=10)
    ax.set_axis_off()
    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()


def _read_image_meta(path: str) -> Tuple[int, int, int]:
    try:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            return 0, 0, 0
        h, w = img.shape[:2]
        c = 1 if img.ndim == 2 else int(img.shape[2])
        return int(w), int(h), int(c)
    except Exception:
        return 0, 0, 0


In [4]:
# /content/uwlfa_gui_cells/cell4_load_repo.py

base_dir = Path("/content") if Path("/content").exists() else Path.cwd()
zip_bytes = _download_repo_zip(OWNER, REPO, BRANCH)
REPO_DIR = _extract_zip_to_tmp(zip_bytes, base_dir)
DB_DIR = REPO_DIR / BASE_REL
assert DB_DIR.exists(), f"Missing path: {DB_DIR}"

DATASETS: Dict[str, List[Dict]] = {}

for sub in sorted([p for p in DB_DIR.iterdir() if p.is_dir()], key=lambda p: p.name.lower()):
    files_out: List[Dict] = []
    for p in sorted(sub.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            w, h, c = _read_image_meta(p.as_posix())
            fname = p.name
            files_out.append(
                {
                    "filename": fname,
                    "path": p.as_posix(),
                    "width": w,
                    "height": h,
                    "channels": c,
                    "row": row_name_from_filename(fname),
                }
            )
    DATASETS[sub.name] = files_out

payload = {
    "repo_dir": str(REPO_DIR),
    "db_dir": str(DB_DIR),
    "folders": [{"name": k, "files": v} for k, v in DATASETS.items()],
}


In [5]:
# /content/uwlfa_gui_cells/cell5_flask_template.py

import datetime
from flask import Flask, jsonify, request, Response  # noqa

app = Flask(__name__)

_today = datetime.date.today()
DATE_FOLDER = f"{_today.month}-{_today.day}-{_today.year}"

TEMPLATE_HTML = r"""
<!doctype html>
<html><head><meta charset="utf-8" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<title>UW-LFA Dataset ROI</title>
<style>
 body{font:14px/1.45 system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif;background:#0b0d12;color:#e6edf3;margin:0}
 .wrap{padding:16px 20px}
 .btn{background:#2a6de0;border:none;color:#fff;padding:10px 14px;border-radius:8px;cursor:pointer;font-weight:600}
 .btn2{background:#334155}
 label{opacity:.8;margin-right:8px}
 select, input[type=text], input[type=number]{background:#0f1623;border:1px solid #263145;color:#e6edf3;border-radius:8px;padding:7px 8px}
 .row{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin:10px 0}
 #imgWrap{position:relative;display:inline-block;border:1px solid #263145;border-radius:10px;overflow:hidden;background:#0f1623;max-width:720px;width:100%}
 #img{display:block;width:100%;height:auto}
 .rect{position:absolute;border:2px dashed #82aaff;background:rgba(130,170,255,.12);pointer-events:none}
 pre{background:#141824;border-radius:10px;padding:12px;overflow:auto}
 .ok{background:#2ea043}
 .panel{border:1px solid #263145;border-radius:10px;padding:12px;background:#0f1623}
 .mt{margin-top:18px}
 .muted{opacity:.8}
 .mono{font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace}
 .loading{opacity:.6}
 #sbrCanvas{width:100%;max-width:960px;height:320px;border:1px solid #263145;border-radius:10px;background:#0b1020;display:none}
 .pill{display:inline-flex;gap:8px;align-items:center;padding:6px 10px;border:1px solid #263145;border-radius:999px;background:#0b1020}
 .grid2{display:grid;grid-template-columns:1fr;gap:10px}
 @media (min-width: 980px){ .grid2{grid-template-columns:1fr 1fr} }
 img{max-width:100%}
</style>
</head>
<body>
<div class="wrap">
  <h2>UW-LFA Dataset ROI</h2>

  <!-- NEW: paragraph explanation under title -->
  <p class="muted" style="max-width:960px;margin:6px 0 14px">
    This tool helps you step through UW-LFA strip images, select a Region of Interest (ROI), rotate and apply grayscale,
    save ROIs by dataset/row, and export a CSV that includes the saved ROI paths plus SBR peak/baseline values.
    Use <b>Select ROI</b> → crop/zoom, then <b>Rotate</b>/<b>Apply Grayscale</b> as needed, <b>Save ROI</b>, and <b>Export CSV</b>.
    The <b>ROI Grid</b> lets you review saved ROIs by date + dataset, and the <b>SBR Profile</b> canvas lets you place the blue/red points
    to compute peak and baseline and write results back into the CSV.
  </p>

  <div class="panel" style="display:flex;gap:10px;align-items:center;flex-wrap:wrap">
    <span class="muted">Saving to:</span>
    <input id="savePath" class="mono" type="text" style="min-width:360px;flex:1" readonly />
    <button id="copyPath" class="btn">Copy</button>
  </div>

  <!-- NEW: dataset description under Saving to -->
  <p class="muted" style="max-width:960px;margin:10px 0 14px">
    Datasets loaded here come from the UW-LFA analysis repository:
    <a href="https://github.com/timrobinson/UW-LFA-Analysis/tree/main/100microliters/Database" target="_blank" rel="noopener noreferrer" style="color:#82aaff">
      https://github.com/timrobinson/UW-LFA-Analysis/tree/main/100microliters/Database
    </a>.
    Each top-level folder represents either a collection date (for example <span class="mono">11-1-23_n=3</span>, where <span class="mono">n</span> is the number of replicate strips)
    or a lossless capture grouping by assay channel (for example <span class="mono">LossLessFormat_N1</span>, <span class="mono">N2</span>, <span class="mono">N3</span>).
    Inside each dataset folder are strip images stored as <span class="mono">.jpg</span> or <span class="mono">.tif/.tiff</span> files.
    Filenames encode the experimental condition: scientific-notation values such as <span class="mono">1e5</span>, <span class="mono">5e6</span>, or <span class="mono">1e7</span>
    indicate target concentration, while labels like <span class="mono">pos</span>, <span class="mono">neg</span>, or <span class="mono">dip</span>
    indicate control conditions; the suffix <span class="mono">n=3</span> denotes the number of replicate strips in that condition.
  </p>

  <p class="muted" style="max-width:960px;margin:0 0 14px"></p>

  <ol class="muted" style="max-width:960px;margin:0 0 18px 18px">
    <li>Select the dataset containing the strip images you want to analyze using the <b>Dataset (CSV name)</b> dropdown, then click <b>Load dataset</b>.</li>
    <li>Click <b>Select ROI</b>, then use your mouse to drag a box over the image to zoom into the region of interest.</li>
    <li>Crop a <b>single test strip</b> that includes the test and control lines, with a small amount of white space before and after the lines. Avoid including any black background—only the analytical portion of the strip should be visible.</li>
    <li>Click <b>Rotate 90°</b> once to orient the strip correctly. This orientation is required for proper alignment of the plots and graphs.</li>
    <li>Click <b>Apply Grayscale</b> to convert the image to a normalized black-and-white representation used for analysis.</li>
    <li>Click <b>Save ROI</b> to store the cropped strip, then click <b>Export CSV</b> to write the updated measurements and metadata for the dataset.</li>
  </ol>

  <div class="panel" style="margin-top:10px">
    <div class="row">
      <label>Dataset (CSV name):</label>
      <select id="ds"></select>
      <button id="load" class="btn">Load dataset</button>

      <label style="margin-left:18px">Image:</label>
      <select id="imgsel"></select>

      <label style="margin-left:18px">Row:</label>
      <input id="rowName" class="mono" type="text" style="width:160px" readonly />
    </div>

    <div id="imgWrap">
      <img id="img" alt="(no image)" />
      <div id="rect" class="rect" style="display:none"></div>
    </div>

    <div class="row">
      <button id="select" class="btn">Select ROI</button>
      <button id="rotate" class="btn">⟳ Rotate 90°</button>
      <button id="applyGray" class="btn btn2">Apply Grayscale</button>
      <button id="reset"  class="btn">Reset View</button>

      <button id="save" class="btn">Save ROI (auto row)</button>
      <button id="export" class="btn ok">Export CSV for Dataset</button>
    </div>

    <div class="panel mt">
      <h3>ROI Grid (by Date + Dataset)</h3>
      <p class="muted" style="max-width:960px;margin:6px 0 12px">
        The ROI Grid provides a consolidated visual review of all Regions of Interest that have been saved for a specific export date and dataset. Each tile in the grid corresponds to a single cropped, rotated, and grayscaled strip image that has been written to disk via <b>Save ROI</b>. This view is intended for validation before quantitative analysis: it lets you confirm that all strips are consistently framed, properly oriented, and free of background artifacts, and that additional images have been successfully added to the dataset.
      </p>
      <ol class="muted" style="max-width:960px;margin:0 0 14px 18px">
        <li><b>Date</b>: Select the export date corresponding to the ROI save session you want to review. Dates are created automatically when ROIs are saved.</li>
        <li><b>↻ Refresh</b>: Reload the list of available export dates from disk after saving or exporting new ROIs.</li>
        <li><b>Dataset</b>: Choose the dataset whose saved ROIs you want to display in the grid.</li>
        <li><b>Plot Grid</b>: Render all saved ROIs for the selected date and dataset into a single grid image.</li>
        <li>To add additional image data to this grid, return to the main image view above, load another image from the same dataset, repeat the ROI selection, rotation, and grayscale steps, then click <b>Save ROI</b> again. Re-plotting the grid will include the newly saved ROI.</li>
      </ol>
      <div class="row">
        <label>Date:</label>
        <select id="dfDate"></select>
        <button id="refreshDfDates" class="btn">↻</button>

        <label style="margin-left:12px">Dataset:</label>
        <select id="gridDataset"></select>

        <label style="margin-left:12px">Dip:</label>
        <select id="gridDip" style="min-width:140px">
          <option value="0" selected>Hide dip</option>
          <option value="1">Show dip</option>
        </select>

        <button id="plotGrid" class="btn">Plot Grid</button>
      </div>
      <img id="dfGrid" alt="ROI grid"
           style="display:none;max-width:960px;width:100%;border:1px solid #263145;border-radius:10px" />
    </div>

    <div class="panel mt">
      <p class="muted" style="max-width:960px;margin:6px 0 12px">
        The SBR Profile section is where quantitative signal analysis happens. Using the cropped and grayscaled ROI saved above, this view plots the intensity profile along the length of the test strip. You interactively place two vertical markers on the curve to define the baseline regions on either side of a positive result. From these points, the tool computes a smoothed peak (green) and a baseline-at-peak value (yellow), which are then written directly back into the dataset CSV for downstream analysis.
      </p>
      <ol class="muted" style="max-width:960px;margin:0 0 14px 18px">
        <li>Select a <b>Row</b> from the dropdown (or provide an index) corresponding to a saved ROI. If no rows appear, make sure you have exported the dataset CSV.</li>
        <li>Click <b>Load SBR</b> to display the intensity profile for the selected strip.</li>
        <li>On the graph, click once to place the <b>blue line</b> just before the positive signal, at the point where the curve flattens into the baseline region.</li>
        <li>Click a second time to place the <b>red line</b> just after the positive signal, again where the curve returns to its flattened baseline.</li>
        <li>The tool automatically computes the median baseline regions, identifies the <b>green peak</b>, calculates the <b>yellow baseline-at-peak</b>, and saves these values into the CSV.</li>
        <li>If the markers were placed incorrectly, click <b>Reset Lines</b> and repeat the process until the baseline and peak correctly bracket the positive result.</li>
      </ol>
      <h3>SBR Profile (Blue + Red → green peak → yellow baseline → save into CSV)</h3>
      <div class="row">
        <label>Row</label>
        <select id="sbrRow" style="min-width:220px"></select>

        <span class="muted mono">or index</span>
        <input id="sbrIndex" type="number" min="0" value="0" style="width:80px" />

        <button id="plotSBR" class="btn">Load SBR</button>
        <button id="resetLines" class="btn btn2">Reset Lines</button>

        <span class="pill"><span class="muted">Blue</span><span id="blueInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Red</span><span id="redInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Peak</span><span id="greenInfo" class="mono">-</span></span>
        <span class="pill"><span class="muted">Baseline@Peak</span><span id="yellowInfo" class="mono">-</span></span>
      </div>

      <canvas id="sbrCanvas"></canvas>
      <div id="sbrMsg" class="muted" style="margin-top:8px"></div>
    </div>

    <div class="panel mt">
      <p class="muted" style="max-width:960px;margin:6px 0 12px">
        The Demo View mirrors the exploratory plots and summary calculations used in the original analysis notebooks. It allows you to visually inspect how the selected SBR peak and baseline propagate into per-strip ratios, row-level statistics, and pooled experiment-level results. This section is intended for validation and interpretation: it shows the underlying strip image, the averaged intensity profile, numerical summaries, and aggregated experiment plots so you can confirm that the chosen baseline and peak definitions behave as expected.
      </p>
      <ol class="muted" style="max-width:960px;margin:0 0 14px 18px">
        <li><b>Run Demo (plots)</b>: Generates diagnostic plots for the currently selected strip using the active blue/red markers. This displays the cropped strip image and the row-averaged intensity profile.</li>
        <li><b>Load Raw Demo Data (JSON)</b>: Loads the full intermediate results dictionary used by the notebook analysis and displays it verbatim. This is useful for debugging or downstream scripting.</li>
        <li><b>Add as Experiment</b>: Commits the current strip’s computed ratio into the experiment pool. Repeated use builds up a multi-strip experiment for statistical aggregation.</li>
        <li><b>Clear Experiments</b>: Removes all accumulated experiment entries and resets pooled statistics and plots.</li>
        <li><b>Row mean ± SEM</b>: Displays the mean signal-to-baseline ratio and standard error for the current row.</li>
        <li><b>Pooled mean ± SEM</b>: Displays the aggregated mean and standard error across all added experiments.</li>
        <li>The <b>imshow(test)</b> panel shows the cropped, processed strip image used for analysis, while the <b>row_averages</b> panel shows the mean intensity profile across the strip width.</li>
        <li>The <b>Experiment summary</b> plot visualizes pooled experiment-level results and updates automatically as experiments are added or cleared.</li>
      </ol>
      <h3>Demo view (matches notebook plots + full results dict)</h3>

      <div class="row">
        <button id="runDemo" class="btn">Run Demo (plots)</button>
        <button id="loadDemoJson" class="btn btn2">Load Raw Demo Data (JSON)</button>
        <button id="addExp" class="btn ok">Add as Experiment</button>
        <button id="clearExp" class="btn btn2">Clear Experiments</button>

        <span class="pill"><span class="muted">Row mean±SEM</span><span id="demoRowStats" class="mono">-</span></span>
        <span class="pill"><span class="muted">Pooled mean±SEM</span><span id="demoPooledStats" class="mono">-</span></span>
      </div>

      <div class="grid2">
        <div class="panel">
          <div class="muted">imshow(test)</div>
          <img id="demoStrip" alt="Demo strip" style="display:none;border:1px solid #263145;border-radius:10px" />
        </div>
        <div class="panel">
          <div class="muted">row_averages = mean(test, axis=0)</div>
          <img id="demoAverage" alt="Average profile" style="display:none;border:1px solid #263145;border-radius:10px" />
        </div>
      </div>

      <div class="panel mt">
        <div class="muted">results = analyze_experiments([R_exp1])</div>
        <pre id="demoJson" class="mono" style="max-height:320px;overflow:auto"></pre>
      </div>

      <img id="expPlot" alt="Experiment summary"
           style="display:none;max-width:960px;width:100%;border:1px solid #263145;border-radius:10px;margin-top:10px" />

      <div class="muted" id="demoMsg" style="margin-top:8px"></div>
    </div>

  </div>

  <div class="panel mt">
    <h3>Final Graph (Graphing.ipynb style)</h3>

    <div class="row">
      <label>Mode:</label>
      <select id="finalMode" style="min-width:260px">
        <option value="combined" selected>Combined (N1–N3)</option>
        <option value="N1">N1 only</option>
        <option value="N2">N2 only</option>
        <option value="N3">N3 only</option>
      </select>

      <span class="muted" style="margin-left:10px">(uses ROI Grid Date/Dataset)</span>

      <label style="margin-left:18px">Source:</label>
      <select id="finalSource" style="min-width:260px">
        <option value="df">Current exported DF (rows)</option>
        <option value="experiments">Experiments list</option>
      </select>
      <button id="plotFinal" class="btn ok">Plot Final Graph</button>
    </div>

    <img id="finalGraph" alt="Final graph"
         style="display:none;max-width:960px;width:100%;border:1px solid #263145;border-radius:10px" />

    <div class="muted" id="finalMsg" style="margin-top:8px"></div>
  </div>

  <pre id="log"></pre>
</div>

<script>
const LOG = (m) => {
  const el = document.getElementById('log');
  el.textContent += m + "\n";
  el.scrollTop = el.scrollHeight;
};
const MSG = (m)=>{document.getElementById('sbrMsg').textContent = m || "";};

async function loadDfFor(dataset, date){
  if(!dataset) return {ok:false, msg:"missing dataset"};
  const payload = { dataset: dataset };
  if(date && date !== '(none)') payload.date = date;

  try{
    const r = await fetch('/load_df', {
      method:'POST',
      headers:{'Content-Type':'application/json'},
      body: JSON.stringify(payload)
    });
    const j = await r.json();
    if(!j.ok){
      LOG(j.msg || 'Load DF: not available yet.');
      return j;
    }
    window.DF_DATASET = dataset;
    LOG(j.msg || `Loaded DF for ${dataset}`);
    return j;
  }catch(e){
    LOG('Load DF failed: ' + e);
    return {ok:false, msg:String(e)};
  }
}

async function fillSavePath(){
  try{
    const r = await fetch('/export_info'); const j = await r.json();
    document.getElementById('savePath').value = j.full;
  }catch(e){}
}
document.getElementById('copyPath').onclick = ()=>{
  const el = document.getElementById('savePath'); el.select(); document.execCommand('copy');
};

const DS = document.getElementById('ds');
const IMGSEL = document.getElementById('imgsel');
const IMG = document.getElementById('img');
const RECT = document.getElementById('rect');
const ROWNAME = document.getElementById('rowName');
const GRID_DS = document.getElementById('gridDataset');

let mouseDown=false, startX=0, startY=0;
let currentDataset = null, currentIndex = null;
let imgBusy = false;
let roiMode=false;

IMG.style.userSelect = 'none';
IMG.style.touchAction = 'none';

function hardResetROI(){
  RECT.style.display='none';
  RECT.style.left = RECT.style.top = '0px';
  RECT.style.width = RECT.style.height = '0px';
  mouseDown = false; roiMode = false;
  IMG.style.cursor = 'default';
}

function setImgSrc() {
  if (!currentDataset || currentIndex===null) return;
  if (imgBusy) return;
  imgBusy = true;
  IMG.classList.add('loading');
  const url = `/image?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}&t=${Date.now()}`;
  const done = ()=>{ imgBusy=false; IMG.classList.remove('loading'); };
  IMG.onload = done; IMG.onerror = done;
  IMG.src = url;
}

async function fetchDatasets(){
  const r = await fetch('/datasets'); const j = await r.json();
  const opts = j.names.map(n=>`<option>${n}</option>`).join('');
  DS.innerHTML = opts;
  GRID_DS.innerHTML = opts;
  LOG("Datasets: " + j.names.join(", "));
}

async function loadDataset(){
  currentDataset = DS.value; currentIndex = null;
  const r = await fetch('/images?dataset=' + encodeURIComponent(currentDataset));
  const j = await r.json();
  if (!j.files.length){ IMGSEL.innerHTML=""; IMG.removeAttribute('src'); ROWNAME.value=""; LOG("No images."); return; }

  IMGSEL.innerHTML = j.files.map((f,i)=>`<option value="${i}">${f.filename} (${f.width}×${f.height})</option>`).join('');
  hardResetROI();
  currentIndex = 0;
  ROWNAME.value = j.files[0].row;
  setImgSrc();
  LOG(`Loaded dataset: ${currentDataset} (${j.files.length} files)`);

  const date = (typeof DFDATES !== 'undefined' && DFDATES && DFDATES.value) ? DFDATES.value : '';
  await loadDfFor(currentDataset, date);
  await fetchDfRows();
}

IMGSEL.onchange = async e => {
  currentIndex = parseInt(e.target.value,10);
  hardResetROI();
  const r = await fetch(`/row_name?dataset=${encodeURIComponent(currentDataset)}&index=${currentIndex}`);
  const j = await r.json();
  ROWNAME.value = j.row || '';
  setImgSrc();
};

document.getElementById('load').onclick = loadDataset;

document.getElementById('select').onclick = () => {
  roiMode = !roiMode;
  RECT.style.display = roiMode ? 'block' : 'none';
  IMG.style.cursor = roiMode ? 'crosshair' : 'default';
  LOG("ROI: " + (roiMode ? "ON" : "OFF"));
};

document.getElementById('reset').onclick  = async () => {
  hardResetROI();
  try{
    const r = await fetch('/reset_view', {method:'POST'}); const j = await r.json(); LOG(j.msg || 'Reset.');
  }catch(e){ LOG('Reset failed: ' + e); }
  setImgSrc();
};

document.getElementById('rotate').onclick = async () => {
  const r = await fetch('/rotate', {method:'POST'}); const j = await r.json(); LOG(j.msg); setImgSrc();
};

document.getElementById('applyGray').onclick = async () => {
  try{
    const r = await fetch('/apply_grayscale', {method:'POST'});
    const j = await r.json();
    LOG(j.msg || "Applied grayscale.");
  }catch(e){
    LOG("Apply grayscale failed: " + e);
  }
  setImgSrc();
};

function clamp(v,min,max){ return Math.max(min, Math.min(max, v)); }
function posInImg(e){
  const r = IMG.getBoundingClientRect();
  const x = clamp(e.clientX - r.left, 0, r.width);
  const y = clamp(e.clientY - r.top , 0, r.height);
  return {x, y, rect: r};
}

IMG.addEventListener('dragstart', e => e.preventDefault());
IMG.addEventListener('mousedown', e=>{
  if(!roiMode) return; e.preventDefault();
  const p = posInImg(e);
  mouseDown = true; startX = p.x; startY = p.y;
  RECT.style.left = startX + 'px'; RECT.style.top  = startY + 'px';
  RECT.style.width = '0px'; RECT.style.height = '0px';
});

window.addEventListener('mousemove', e=>{
  if(!roiMode || !mouseDown) return;
  const p = posInImg(e);
  const left = Math.min(startX, p.x), top = Math.min(startY, p.y);
  const w = Math.abs(p.x - startX), h = Math.abs(p.y - startY);
  RECT.style.left=left+'px'; RECT.style.top=top+'px'; RECT.style.width=w+'px'; RECT.style.height=h+'px';
});

window.addEventListener('mouseup', async e=>{
  if(!roiMode || !mouseDown) return; mouseDown = false;
  const r = IMG.getBoundingClientRect();
  const curr = posInImg(e);
  const left = Math.min(startX, curr.x), top  = Math.min(startY, curr.y);
  const w = Math.abs(curr.x - startX), h = Math.abs(curr.y - startY);
  if (w < 3 || h < 3) { LOG("Selection too small."); return; }
  const nx = left / r.width, ny = top  / r.height, nw = w / r.width, nh = h / r.height;
  const tw = Math.round(r.width), th = Math.round(r.height);
  const resp = await fetch('/apply_crop_zoom', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({nx, ny, nw, nh, tw, th})
  });
  const j = await resp.json(); LOG(j.msg); setImgSrc();
});

document.getElementById('save').onclick = async () => {
  if (!currentDataset || currentIndex===null) return;
  const r = await fetch('/save_roi', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset, index: currentIndex })
  });
  const j = await r.json(); LOG(j.msg);
};

document.getElementById('export').onclick = async () => {
  if (!currentDataset) return;
  const r = await fetch('/export_df', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ dataset: currentDataset })
  });
  const j = await r.json();
  LOG(j.msg);

  const date = (typeof DFDATES !== 'undefined' && DFDATES && DFDATES.value) ? DFDATES.value : '';
  await loadDfFor(currentDataset, date);
  await fetchDfRows();
};

// dates + grid
const DFGRID  = document.getElementById('dfGrid');
const DFDATES = document.getElementById('dfDate');

async function fetchDfDates(){
  const r = await fetch('/exports'); const j = await r.json();
  if (!j.dates.length){ DFDATES.innerHTML = '<option>(none)</option>'; DFGRID.style.display='none'; return; }
  DFDATES.innerHTML = j.dates.map(d=>`<option>${d}</option>`).join('');
}
document.getElementById('refreshDfDates').onclick = fetchDfDates;

document.getElementById('plotGrid').onclick = () => {
  const date = DFDATES.value;
  const ds = GRID_DS.value;
  const dip = (document.getElementById('gridDip')?.value || '0');
  if(!date || date === '(none)'){ LOG('No export dates found.'); return; }
  if(!ds){ LOG('Pick a dataset.'); return; }
  DFGRID.src = `/analyze_grid?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&include_dip=${encodeURIComponent(dip)}&t=${Date.now()}`;
  DFGRID.style.display = 'block';
};

// ----------------- interactive SBR -----------------
const canvas = document.getElementById('sbrCanvas');
const ctx = canvas.getContext('2d');
const blueInfo = document.getElementById('blueInfo');
const redInfo = document.getElementById('redInfo');
const greenInfo = document.getElementById('greenInfo');
const yellowInfo = document.getElementById('yellowInfo');

const sbrRowSel = document.getElementById('sbrRow');
let sbrProfile = null;
let sbrN = 0;
let xBlue = null;
let xRed = null;
let xPeak = null;
let peakVal = null;
let baselineVal = null;
let baselineXHalf = null;
let sbrRowName = null;

let plotGeom = null;

function setCanvasSize(){
  const rect = canvas.getBoundingClientRect();
  const dpr = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(rect.width * dpr));
  canvas.height = Math.max(1, Math.floor(rect.height * dpr));
  ctx.setTransform(dpr,0,0,dpr,0,0);
}

function resetLines(){
  xBlue = null; xRed = null; xPeak = null;
  peakVal = null;
  baselineVal = null;
  baselineXHalf = null;
  blueInfo.textContent = "-";
  redInfo.textContent = "-";
  greenInfo.textContent = "-";
  yellowInfo.textContent = "-";
  MSG("Click two points on the graph (x must be >=5 and <= n-6).");
  drawSBR();
}
document.getElementById('resetLines').onclick = resetLines;

async function fetchDfRows(){
  try{
    const r = await fetch('/df_rows'); const j = await r.json();
    if(!j.ok){ sbrRowSel.innerHTML = '<option>(export first)</option>'; return; }
    const rows = j.rows || [];
    if(!rows.length){ sbrRowSel.innerHTML = '<option>(no rows)</option>'; return; }
    sbrRowSel.innerHTML = rows.map(x=>`<option value="${x}">${x}</option>`).join('');
  }catch(e){
    sbrRowSel.innerHTML = '<option>(error)</option>';
  }
}

function niceNum(range, round) {
  const exponent = Math.floor(Math.log10(range || 1));
  const fraction = range / Math.pow(10, exponent);
  let niceFraction;
  if (round) {
    if (fraction < 1.5) niceFraction = 1;
    else if (fraction < 3) niceFraction = 2;
    else if (fraction < 7) niceFraction = 5;
    else niceFraction = 10;
  } else {
    if (fraction <= 1) niceFraction = 1;
    else if (fraction <= 2) niceFraction = 2;
    else if (fraction <= 5) niceFraction = 5;
    else niceFraction = 10;
  }
  return niceFraction * Math.pow(10, exponent);
}

function computeTicks(min, max, maxTicks = 5) {
  const range = niceNum(max - min, false);
  const step = niceNum(range / Math.max(1, (maxTicks - 1)), true);
  const graphMin = Math.floor(min / step) * step;
  const graphMax = Math.ceil(max / step) * step;

  const ticks = [];
  for (let v = graphMin; v <= graphMax + 0.5 * step; v += step) ticks.push(v);
  return { ticks, graphMin, graphMax, step };
}

function formatTick(v, step){
  const absStep = Math.abs(step || 0);
  let decimals = 0;
  if (absStep > 0) {
    decimals = Math.max(0, -Math.floor(Math.log10(absStep)) + 1);
    decimals = Math.min(decimals, 6);
  }
  let s = v.toFixed(decimals);
  s = s.replace(/\.?0+$/, '');
  return s;
}

function drawSBR(){
  if (!sbrProfile || !sbrProfile.length){
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }
  canvas.style.display = 'block';
  setCanvasSize();

  const rect = canvas.getBoundingClientRect();
  const W = rect.width;
  const H = rect.height;
  ctx.clearRect(0,0,W,H);

  const padL = 62, padR = 18, padT = 34, padB = 46;

  let yMin = Infinity, yMax = -Infinity;
  for (const v of sbrProfile){ if (v<yMin) yMin=v; if (v>yMax) yMax=v; }
  if (!isFinite(yMin) || !isFinite(yMax)){ yMin=0; yMax=1; }

  if (baselineVal !== null && isFinite(baselineVal)){
    yMin = Math.min(yMin, baselineVal);
    yMax = Math.max(yMax, baselineVal);
  }

  const yPad = (yMax - yMin) * 0.08 || 0.01;
  yMin -= yPad; yMax += yPad;

  const maxX = sbrN - 1;
  const xTicks = [];
  const xStep = 50;
  for (let x=0; x<=maxX; x+=xStep) xTicks.push(x);
  if (xTicks[xTicks.length-1] !== maxX) xTicks.push(maxX);

  const yt = computeTicks(yMin, yMax, 5);

  const xToPx = (x) => padL + (x / (sbrN - 1)) * (W - padL - padR);
  const yToPx = (y) => (H - padB) - ((y - yt.graphMin) / (yt.graphMax - yt.graphMin + 1e-9)) * (H - padT - padB);

  plotGeom = { padL, padR, padT, padB, W, H, xToPx, yToPx };

  ctx.lineWidth = 1;
  ctx.strokeStyle = "rgba(230,237,243,0.18)";
  ctx.font = "12px ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace";
  ctx.fillStyle = "#e6edf3";

  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  for (const xtv of xTicks){
    const x = xToPx(xtv);
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
    ctx.fillText(String(xtv), x, H-padB+8);
  }

  ctx.textAlign = "right";
  ctx.textBaseline = "middle";
  for (const ytv of yt.ticks){
    const y = yToPx(ytv);
    ctx.beginPath();
    ctx.moveTo(padL, y);
    ctx.lineTo(W-padR, y);
    ctx.stroke();
    ctx.fillText(formatTick(ytv, yt.step), padL-10, y);
  }

  ctx.strokeStyle = "rgba(230,237,243,0.35)";
  ctx.lineWidth = 1.25;
  ctx.beginPath();
  ctx.rect(padL, padT, (W-padL-padR), (H-padT-padB));
  ctx.stroke();

  ctx.fillStyle = "#e6edf3";
  ctx.font = "600 16px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  const title = `SBR – ${(window.DF_DATASET || "")} – row ${sbrRowName || ""}`;
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText(title, W/2, 8);

  ctx.fillStyle = "#e6edf3";
  ctx.font = "13px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Position along strip", (padL + (W-padR))/2, H-28);

  ctx.save();
  ctx.translate(18, (padT + (H-padB))/2);
  ctx.rotate(-Math.PI/2);
  ctx.textAlign = "center";
  ctx.textBaseline = "top";
  ctx.fillText("Intensity (smoothed)", 0, 0);
  ctx.restore();

  function shadeWindow(ix, rgba){
    if (ix === null) return;
    const left = xToPx(ix - 5);
    const right = xToPx(ix + 5);
    ctx.fillStyle = rgba;
    ctx.fillRect(left, padT, (right-left), (H-padB-padT));
  }
  shadeWindow(xBlue, "rgba(59,130,246,0.12)");
  shadeWindow(xRed,  "rgba(239,68,68,0.12)");

  if (baselineVal !== null && isFinite(baselineVal)){
    const yb = yToPx(baselineVal);
    ctx.strokeStyle = "#eab308";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(padL, yb);
    ctx.lineTo(W-padR, yb);
    ctx.stroke();
  }

  ctx.strokeStyle = "#9bbcff";
  ctx.lineWidth = 2;
  ctx.beginPath();
  for (let i=0;i<sbrN;i++){
    const x = xToPx(i);
    const y = yToPx(sbrProfile[i]);
    if (i===0) ctx.moveTo(x,y); else ctx.lineTo(x,y);
  }
  ctx.stroke();

  function drawV(ix, color){
    const x = xToPx(ix);
    ctx.strokeStyle = color;
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.moveTo(x, padT);
    ctx.lineTo(x, H-padB);
    ctx.stroke();
  }
  if (xBlue !== null) drawV(xBlue, "#3b82f6");
  if (xRed !== null)  drawV(xRed,  "#ef4444");
  if (xPeak !== null) drawV(xPeak, "#22c55e");

  ctx.font = "12px system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,sans-serif";
  ctx.textAlign = "left";
  ctx.textBaseline = "top";
  let lx = padL + 8, ly = padT + 8;
  if (xBlue !== null){
    ctx.fillStyle = "#3b82f6"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Blue @ ${xBlue} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xRed !== null){
    ctx.fillStyle = "#ef4444"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Red @ ${xRed} (±5)`, lx+14, ly);
    ly += 18;
  }
  if (xPeak !== null){
    ctx.fillStyle = "#22c55e"; ctx.fillRect(lx, ly+4, 10, 2);
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Peak @ ${xPeak}`, lx+14, ly);
    ly += 18;
  }
  if (baselineVal !== null && isFinite(baselineVal)){
    ctx.fillStyle = "#eab308"; ctx.fillRect(lx, ly+4, 10, 2);
    const xTxt = (baselineXHalf !== null) ? ` (x=${baselineXHalf})` : "";
    ctx.fillStyle = "#e6edf3"; ctx.fillText(`Baseline@Peak = ${Number(baselineVal).toFixed(6)}${xTxt}`, lx+14, ly);
  }
}

function pxToIndex(clientX){
  if (!plotGeom) return 0;
  const rect = canvas.getBoundingClientRect();
  const { padL, padR, W } = plotGeom;
  const x = Math.max(padL, Math.min(W-padR, clientX - rect.left));
  const t = (x - padL) / (W - padL - padR);
  return Math.round(t * (sbrN - 1));
}

function validateIndex(ix){
  if (ix < 5 || ix > (sbrN - 6)){
    return {ok:false, msg:`Pick a different point: need 5 points on each side (valid: 5..${sbrN-6}). You picked ${ix}.`};
  }
  return {ok:true, msg:""};
}

canvas.addEventListener('click', async (e)=>{
  if (!sbrProfile) return;
  const ix = pxToIndex(e.clientX);
  const v = validateIndex(ix);
  if (!v.ok){ MSG(v.msg); return; }

  if (xBlue === null){
    xBlue = ix;
    blueInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    MSG("Blue set. Click second point for Red.");
    drawSBR();
    return;
  }
  if (xRed === null){
    xRed = ix;
    redInfo.textContent = `${ix} (window ${ix-5}-${ix+5})`;
    drawSBR();
    MSG("Red set. Computing medians + peak + baseline, updating CSV...");

    const row = sbrRowSel.value || "";
    const idx = document.getElementById('sbrIndex').value || '0';

    const r = await fetch('/update_medians', {
      method:'POST', headers:{'Content-Type':'application/json'},
      body: JSON.stringify({ row: row, index: parseInt(idx,10), x_blue: xBlue, x_red: xRed })
    });
    const j = await r.json();
    if (j.ok){
      xPeak = j.peak_x;
      peakVal = j.peak_value;
      baselineVal = j.baseline_value;
      baselineXHalf = j.baseline_xhalf;

      greenInfo.textContent = `${xPeak} (val ${Number(peakVal).toFixed(6)})`;
      yellowInfo.textContent = `${Number(baselineVal).toFixed(6)} (x ${baselineXHalf})`;
      drawSBR();
    }
    MSG(j.msg || (j.ok ? "Updated CSV." : "Failed."));
    return;
  }
  MSG("You already selected 2 points. Click Reset Lines to choose different ones.");
});

document.getElementById('plotSBR').onclick = async () => {
  // Make sure the active DF matches the dataset we're analyzing.
  const date = (typeof DFDATES !== 'undefined' && DFDATES && DFDATES.value) ? DFDATES.value : '';
  await loadDfFor(currentDataset, date);

  const row = sbrRowSel.value || "";
  const idx = document.getElementById('sbrIndex').value || '0';
  const url = `/sbr_profile_json?row=${encodeURIComponent(row)}&index=${encodeURIComponent(idx)}&t=${Date.now()}`;
  const r = await fetch(url);
  const j = await r.json();
  if (!j.ok){
    MSG(j.msg || "Failed to load SBR. Did you click Export CSV first?");
    canvas.style.display = 'none';
    plotGeom = null;
    return;
  }
  window.DF_DATASET = j.dataset || "";
  sbrProfile = j.profile;
  sbrN = j.n;
  sbrRowName = j.row_name;
  resetLines();
};

// ----------------- Demo viewer -----------------
const demoMsg = document.getElementById('demoMsg');
const demoStrip = document.getElementById('demoStrip');
const demoAverage = document.getElementById('demoAverage');
const demoJson = document.getElementById('demoJson');
const demoRowStats = document.getElementById('demoRowStats');
const demoPooledStats = document.getElementById('demoPooledStats');
const expPlot = document.getElementById('expPlot');

function setDemoMsg(m){ demoMsg.textContent = m || ""; }

async function runDemoPlots(){
  if (xBlue === null || xRed === null){
    setDemoMsg("Pick Blue and Red points on the SBR graph first.");
    return;
  }
  const row = sbrRowSel.value || "";
  const idx = document.getElementById('sbrIndex').value || '0';

  const j = await (await fetch(`/demo_full_json?row=${encodeURIComponent(row)}&index=${encodeURIComponent(idx)}&x_blue=${xBlue}&x_red=${xRed}&t=${Date.now()}`)).json();
  if (!j.ok){
    setDemoMsg(j.msg || "Demo failed.");
    return;
  }

  demoRowStats.textContent = `${Number(j.demo.row_ratio_mean).toFixed(4)} ± ${Number(j.demo.row_ratio_sem).toFixed(4)} (n=${j.demo.row_ratio_n})`;

  demoStrip.src = `/demo_strip.png?row=${encodeURIComponent(j.row_name)}&t=${Date.now()}`;
  demoAverage.src = `/demo_average.png?row=${encodeURIComponent(j.row_name)}&t=${Date.now()}`;

  demoStrip.style.display = 'block';
  demoAverage.style.display = 'block';

  setDemoMsg("Demo plots updated.");
}

document.getElementById('runDemo').onclick = runDemoPlots;

document.getElementById('loadDemoJson').onclick = async ()=>{
  if (xBlue === null || xRed === null){
    setDemoMsg("Pick Blue and Red points first.");
    return;
  }
  const row = sbrRowSel.value || "";
  const idx = document.getElementById('sbrIndex').value || '0';
  const r = await fetch(`/demo_full_json?row=${encodeURIComponent(row)}&index=${encodeURIComponent(idx)}&x_blue=${xBlue}&x_red=${xRed}&t=${Date.now()}`);
  const j = await r.json();
  if (!j.ok){
    setDemoMsg(j.msg || "Demo JSON failed.");
    return;
  }
  demoJson.textContent = JSON.stringify(j, null, 2);
  setDemoMsg("Raw demo JSON loaded (includes results.R_all).");
};

document.getElementById('addExp').onclick = async ()=>{
  if (xBlue === null || xRed === null){
    setDemoMsg("Pick Blue and Red points first.");
    return;
  }
  const row = sbrRowSel.value || "";
  const r = await fetch('/experiment_add', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({ row: row, x_blue: xBlue, x_red: xRed })
  });
  const out = await r.json();
  if (!out.ok){
    setDemoMsg(out.msg || "Failed to add experiment.");
    return;
  }
  demoPooledStats.textContent = `${Number(out.R_best).toFixed(4)} ± ${Number(out.SE_best).toFixed(4)} (N=${out.N_total})`;
  expPlot.src = `/experiment_plot.png?t=${Date.now()}`;
  expPlot.style.display = 'block';
  setDemoMsg(out.msg || "Experiment added.");
};

document.getElementById('clearExp').onclick = async ()=>{
  const r = await fetch('/experiment_clear', {method:'POST'});
  const j = await r.json();
  demoPooledStats.textContent = "-";
  expPlot.style.display = 'none';
  setDemoMsg(j.msg || "Cleared.");
};

// ----------------- Final graph -----------------
const finalGraph = document.getElementById('finalGraph');
const finalMsg = document.getElementById('finalMsg');


document.getElementById('plotFinal').onclick = () => {
  const mode = (document.getElementById('finalMode')?.value || 'combined').toUpperCase();
  const srcLegacy = document.getElementById('finalSource').value || "df";

  const date = (typeof DFDATES !== 'undefined' ? DFDATES.value : "") || "";
  const ds = (typeof GRID_DS !== 'undefined' ? GRID_DS.value : "") || (currentDataset || "");

  if (!date || date === '(none)') {
    finalMsg.textContent = "Pick an export Date (ROI Grid section) first.";
    return;
  }
  if (!ds) {
    finalMsg.textContent = "Pick a Dataset (ROI Grid section) first.";
    return;
  }

  finalMsg.textContent = "Rendering...";

  let url = "";
  if (mode === 'COMBINED') {
    url = `/final_graph_combined.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&t=${Date.now()}`;
  } else if (mode === 'N1' || mode === 'N2' || mode === 'N3') {
    url = `/final_graph_selected.png?date=${encodeURIComponent(date)}&dataset=${encodeURIComponent(ds)}&mode=${encodeURIComponent(mode)}&t=${Date.now()}`;
  } else {
    url = `/final_graph.png?source=${encodeURIComponent(srcLegacy)}&t=${Date.now()}`;
  }

  finalGraph.src = url;
  finalGraph.style.display = 'block';
  finalGraph.onload = () => { finalMsg.textContent = ""; };
  finalGraph.onerror = () => {
    finalMsg.textContent =
      "Failed to load final graph. Ensure the endpoint exists and CSV exports exist for the selected date/dataset.";
  };
};

// ----------------- init -----------------
function init() {
  fillSavePath();
  fetchDatasets()
    .then(fetchDfDates)
    .then(fetchDfRows)
    .catch(e => LOG("Init error: " + e));
}
window.addEventListener('load', init);
window.addEventListener('resize', () => { if (sbrProfile) drawSBR(); });

</script>
</body></html>
"""


In [6]:
# /content/uwlfa_gui_cells/cell6_state_and_helpers.py

# ----------------- server state -----------------
CURRENT_IMAGE = None
ORIGINAL_IMAGE = None

TARGET_H, TARGET_W = 70, 270

# Split inversion so preview can stay looking the way you like,
# while analysis matches 100microliters.ipynb logic.
DISPLAY_INVERT_INTENSITY = False  # affects /image preview only
ANALYSIS_INVERT_INTENSITY = True  # affects ROI saving + SBR + demo math (peak not minimum)

# Back-compat alias (do not use for new logic)
INVERT_INTENSITY = ANALYSIS_INVERT_INTENSITY

ROI_BY_DATASET_ROW: Dict[Tuple[str, str], np.ndarray] = {}
ROI_SAVE_COUNT: Dict[Tuple[str, str], int] = {}
ROI_LATEST_PATH: Dict[Tuple[str, str], Path] = {}
ROI_LATEST_SHAPE: Dict[Tuple[str, str], Tuple[int, int]] = {}

DF_GRAY: Optional["pd.DataFrame"] = None
DF_DATASET: Optional[str] = None
DF_CSV_PATH: Optional[Path] = None

EXP_R_LIST: List[np.ndarray] = []
EXP_LABELS: List[str] = []

# =========================
# Graphing.ipynb-style helpers
# =========================

# Rows that are not part of the Graphing.ipynb final curve
EXCLUDE_ROWS_GRAPH_FINAL = {"cc", "k", "dip", "dips"}

# Canonical row order used by Graphing.ipynb
GRAPH_ROW_ORDER = ["neg", "1e5", "1.5e5", "5e5", "1e6", "5e6", "1e7", "pos"]


def _normalize_row_label(row: str) -> str:
    r = (row or "").strip().lower()
    if r == "dips":
        r = "dip"
    if r.startswith("neg"):
        r = "neg"
    if r.startswith("pos"):
        r = "pos"
    return r


def _replicate_from_dataset_name(ds_raw: str) -> Optional[str]:
    s = (ds_raw or "").strip().lower()

    m = re.search(r"_n *= *([123])", s)
    if m:
        return f"N{m.group(1)}"

    m = re.search(r"(^|[^0-9a-z])n([123])([^0-9a-z]|$)", s)
    if m:
        return f"N{m.group(2)}"

    return None


def _row_to_bacterial_load(row: str) -> Optional[float]:
    r = _normalize_row_label(row)
    if not r:
        return None
    if r in {"neg", "negative"}:
        return 0.0
    if r in {"pos", "positive"}:
        return 1e9
    if r in EXCLUDE_ROWS_GRAPH_FINAL:
        return None

    # Backslash-free equivalent of: r"^(\d+(?:\.\d+)?)e(\d+)$"
    m = re.match(r"^([0-9]+(?:[.][0-9]+)?)e([0-9]+)$", r)
    if m:
        base = float(m.group(1))
        exp = float(m.group(2))
        return float(base * (10**exp))

    return None


def _collect_df_graph_points() -> Tuple[List[float], List[float], List[float], List[str]]:
    if DF_GRAY is None or DF_GRAY.empty:
        return [], [], [], []

    loads: List[float] = []
    means: List[float] = []
    sems: List[float] = []
    labels: List[str] = []

    for row_name in DF_GRAY.index.tolist():
        row_raw = str(row_name)
        row_norm = _normalize_row_label(row_raw)

        bl = _row_to_bacterial_load(row_norm)
        if bl is None:
            continue

        try:
            mean_v = DF_GRAY.loc[row_raw].get("row_ratio_mean", "")
            sem_v = DF_GRAY.loc[row_raw].get("row_ratio_sem", "")
        except Exception:
            continue

        if mean_v in ("", None) or sem_v in ("", None):
            continue

        try:
            mean_f = float(mean_v)
            sem_f = float(sem_v) if str(sem_v).strip() != "" else float("nan")
            if not np.isfinite(sem_f):
                sem_f = 0.0
        except Exception:
            continue

        if not np.isfinite(mean_f):
            continue

        loads.append(float(bl))
        means.append(float(mean_f))
        sems.append(float(sem_f))
        labels.append(row_norm)

    order = np.argsort(np.array(loads, dtype=float)).tolist()
    loads = [loads[i] for i in order]
    means = [means[i] for i in order]
    sems = [sems[i] for i in order]
    labels = [labels[i] for i in order]
    return loads, means, sems, labels


def _compute_threshold_from_df(
    df: "pd.DataFrame",
    loads: List[float],
    means: List[float],
    sems: List[float],
    labels: List[str],
) -> Optional[float]:
    neg_idx = None
    for i, lab in enumerate(labels):
        if (lab or "").strip().lower() in {"neg", "negative"}:
            neg_idx = i
            break
    if neg_idx is None:
        for i, bl in enumerate(loads):
            if float(bl) == 0.0:
                neg_idx = i
                break
    if neg_idx is None:
        return None

    neg_mean = float(means[neg_idx])
    neg_sem = float(sems[neg_idx])

    try:
        n_val = df.loc[labels[neg_idx]].get("row_ratio_n", "")
        n = int(n_val)
    except Exception:
        n = 0

    if n and n > 1 and np.isfinite(neg_sem):
        neg_std = float(neg_sem * np.sqrt(float(n)))
    else:
        neg_std = float(neg_sem) if np.isfinite(neg_sem) else 0.0

    return float(neg_mean + 3.0 * neg_std)


def _compute_threshold_from_neg(
    loads: List[float], means: List[float], sems: List[float], labels: List[str]
) -> Optional[float]:
    if DF_GRAY is None or DF_GRAY.empty:
        return None
    return _compute_threshold_from_df(DF_GRAY, loads, means, sems, labels)


def _final_graph_png_bytes() -> bytes:
    loads, means, sems, labels = _collect_df_graph_points()
    if not loads:
        return _png_message(
            "Final graph: no plotted points.\n Export CSV + compute row_ratio_mean/sem first "
            "(pick Blue/Red on SBR, then Run Demo)."
        )

    thr = _compute_threshold_from_neg(loads, means, sems, labels)

    left_x, left_y, left_e = [], [], []
    right_x, right_y, right_e = [], [], []

    for bl, m, e in zip(loads, means, sems):
        if float(bl) <= 0.0:
            left_x.append(float(bl))
            left_y.append(float(m))
            left_e.append(float(e))
        else:
            right_x.append(float(bl))
            right_y.append(float(m))
            right_e.append(float(e))

    y_all = np.array(means, dtype=float)
    y_min = float(np.nanmin(y_all)) if y_all.size else 0.99
    y_max = float(np.nanmax(y_all)) if y_all.size else 1.05
    if thr is not None and np.isfinite(thr):
        y_min = min(y_min, float(thr))
        y_max = max(y_max, float(thr))
    y_pad = (y_max - y_min) * 0.10 if (y_max > y_min) else 0.05
    y_lo = max(0.0, y_min - y_pad)
    y_hi = y_max + y_pad

    f, axes = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [1, 6]}, figsize=(9.6, 3.8)
    )
    ax0, ax1 = axes[0], axes[1]
    ax1.yaxis.tick_right()

    if right_x:
        ax1.set_xscale("log")

    ax0.set_ylim(y_lo, y_hi)
    ax1.set_ylim(y_lo, y_hi)

    ax0.set_xlim(-0.9, 1.0)
    ax0.set_xticks([0.0])
    ax0.set_xticklabels(["neg"])
    ax0.set_ylabel("Signal to baseline ratio", fontsize=12)

    ax0.yaxis.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax0.xaxis.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax1.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax1.grid(which="minor", alpha=0.22, linestyle=":", linewidth=0.6, zorder=0)
    ax1.minorticks_on()

    s = 45
    if left_x:
        ax0.scatter(
            left_x,
            left_y,
            facecolors="white",
            edgecolor="deepskyblue",
            linewidths=1,
            zorder=3,
            s=s,
        )
        ax0.errorbar(
            left_x,
            left_y,
            yerr=left_e,
            fmt="none",
            ecolor="deepskyblue",
            capsize=4,
            zorder=2,
        )

    if right_x:
        ax1.scatter(
            right_x,
            right_y,
            facecolors="white",
            edgecolor="deepskyblue",
            linewidths=1,
            zorder=3,
            s=s,
            label="UI-derived",
        )
        ax1.errorbar(
            right_x,
            right_y,
            yerr=right_e,
            fmt="none",
            ecolor="deepskyblue",
            capsize=4,
            zorder=2,
        )

    if thr is not None and np.isfinite(thr):
        ax1.plot(
            [min(right_x) if right_x else 1e5, max(right_x) if right_x else 1e9],
            [thr, thr],
            linestyle="dashed",
            c="k",
            alpha=0.5,
            label="Positivity threshold",
        )
        ax0.plot([-1, 1], [thr, thr], linestyle="dashed", c="k", alpha=0.5)

    ds_title = DF_DATASET or ""
    ax1.set_xlabel("S. pyogenes concentration in saliva [CFU/mL]", fontsize=12)
    f.suptitle(f"{ds_title}", fontsize=12, y=0.98)

    if right_x:
        xmin = min(right_x)
        xmax = max(right_x)
        if xmin > 0:
            ax1.set_xlim(max(1e-3, 0.5 * xmin), 2.0 * xmax)

    ax1.legend(facecolor="white", framealpha=0.9, loc="lower left")
    plt.subplots_adjust(wspace=0.05, hspace=0)

    buf = io.BytesIO()
    plt.tight_layout()
    f.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(f)
    buf.seek(0)
    return buf.getvalue()


def _load_export_csv_for_dataset(date: str, ds_raw: str) -> Optional["pd.DataFrame"]:
    try:
        ds = sanitize_name(ds_raw)
        path = EXPORT_ROOT / date / f"{ds}.csv"
        if not path.exists():
            return None
        df = pd.read_csv(path)
        if "row" not in df.columns:
            return None
        df["row"] = df["row"].astype(str).map(_normalize_row_label)
        return df.set_index("row", drop=True)
    except Exception:
        return None


def _resolve_triplet_datasets(date: str, selected_ds_raw: str) -> Dict[str, Optional[str]]:
    want_lossless = "losslessformat" in (selected_ds_raw or "").lower()

    candidates: List[str] = []
    for ds in DATASETS.keys():
        is_lossless = "losslessformat" in ds.lower()
        if want_lossless != is_lossless:
            continue
        rep = _replicate_from_dataset_name(ds)
        if rep in {"N1", "N2", "N3"}:
            candidates.append(ds)

    base = EXPORT_ROOT / date

    def has_export(ds_name: str) -> bool:
        return (base / f"{sanitize_name(ds_name)}.csv").exists()

    triplet: Dict[str, Optional[str]] = {"N1": None, "N2": None, "N3": None}

    for rep in ["N1", "N2", "N3"]:
        rep_cands = [ds for ds in candidates if _replicate_from_dataset_name(ds) == rep]
        rep_exported = [ds for ds in rep_cands if has_export(ds)]

        if selected_ds_raw in rep_exported:
            triplet[rep] = selected_ds_raw
        elif rep_exported:
            triplet[rep] = sorted(rep_exported)[0]
        elif selected_ds_raw in rep_cands:
            triplet[rep] = selected_ds_raw
        elif rep_cands:
            triplet[rep] = sorted(rep_cands)[0]

    return triplet


def _collect_combined_graph_points(
    date: str, selected_ds_raw: str
) -> Tuple[List[float], List[float], List[float], List[str], Optional[float]]:
    triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=selected_ds_raw)

    dfs: Dict[str, pd.DataFrame] = {}
    for rep, ds_raw in triplet.items():
        if not ds_raw:
            continue
        df = _load_export_csv_for_dataset(date, ds_raw)
        if df is None or df.empty:
            continue
        if "row_ratio_mean" not in df.columns:
            continue
        dfs[rep] = df

    if not dfs:
        return [], [], [], [], None

    row_set: set = set()
    for df in dfs.values():
        for r in df.index.tolist():
            rr = _normalize_row_label(str(r))
            if rr in EXCLUDE_ROWS_GRAPH_FINAL:
                continue
            row_set.add(rr)

    rows = [r for r in GRAPH_ROW_ORDER if r in row_set]
    rows.extend(sorted([r for r in row_set if r not in rows]))

    mat: List[List[float]] = []
    for r in rows:
        vals: List[float] = []
        for rep in ["N1", "N2", "N3"]:
            df = dfs.get(rep)
            if df is None:
                vals.append(float("nan"))
                continue
            try:
                v = df.loc[r].get("row_ratio_mean", float("nan"))
            except Exception:
                v = float("nan")
            try:
                vals.append(float(v))
            except Exception:
                vals.append(float("nan"))
        mat.append(vals)

    arr = np.array(mat, dtype=float)
    avg = np.nanmean(arr, axis=1)
    stnd = np.nanstd(arr, axis=1, ddof=1)
    err = stnd / 3.0

    loads: List[float] = []
    means: List[float] = []
    errs: List[float] = []
    labels: List[str] = []

    for r, a, e in zip(rows, avg, err):
        bl = _row_to_bacterial_load(r)
        if bl is None or (not np.isfinite(a)):
            continue
        loads.append(float(bl))
        means.append(float(a))
        errs.append(float(e) if np.isfinite(e) else 0.0)
        labels.append(r)

    thr = None
    try:
        if "neg" in rows:
            i = rows.index("neg")
            if np.isfinite(avg[i]):
                s = float(stnd[i]) if np.isfinite(stnd[i]) else 0.0
                thr = float(avg[i] + 3.0 * s)
    except Exception:
        thr = None

    order = np.argsort(np.array(loads, dtype=float)).tolist()
    loads = [loads[i] for i in order]
    means = [means[i] for i in order]
    errs = [errs[i] for i in order]
    labels = [labels[i] for i in order]

    return loads, means, errs, labels, thr


def _final_graph_combined_png_bytes(date: str, selected_ds_raw: str) -> bytes:
    loads, means, errs, labels, thr = _collect_combined_graph_points(
        date=date, selected_ds_raw=selected_ds_raw
    )
    if not loads:
        return _png_message(
            "Combined graph: no plotted points.\n Export CSVs for N1/N2/N3 "
            "(and run demo for rows), then retry."
        )

    left_x, left_y, left_e = [], [], []
    right_x, right_y, right_e = [], [], []

    for bl, m, e in zip(loads, means, errs):
        if float(bl) <= 0.0:
            left_x.append(float(bl))
            left_y.append(float(m))
            left_e.append(float(e))
        else:
            right_x.append(float(bl))
            right_y.append(float(m))
            right_e.append(float(e))

    lim = float(np.nanmax(np.array(means, dtype=float))) if means else 1.2
    lim = max(1.2, lim + 0.2)

    f, axes = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [1, 6]}, figsize=(9.6, 3.8)
    )
    ax0, ax1 = axes[0], axes[1]
    ax1.yaxis.tick_right()
    ax1.set_xscale("log")

    ax1.set_ylim(0.99, lim)
    ax0.set_ylim(0.99, lim)
    ax0.set_xlim(-0.9, 1)

    ax0.yaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax0.xaxis.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="major", color="#DDDDDD", zorder=0, linewidth=0.8)
    ax1.grid(which="minor", color="#EEEEEE", zorder=0, linewidth=0.5)
    ax1.minorticks_on()

    ax0.set_xticks([0])
    ax0.set_xticklabels(["neg"])
    ax0.set_ylabel("Signal to baseline ratio", fontsize=14)

    color1 = "deepskyblue"
    if left_x:
        ax0.scatter(left_x, left_y, color=color1, alpha=1)
        ax0.errorbar(
            left_x,
            left_y,
            yerr=left_e,
            color=color1,
            alpha=1,
            fmt="none",
            capsize=4,
        )

    if right_x:
        ax1.scatter(right_x, right_y, color=color1, alpha=1, label="avg (N1–N3)")
        ax1.errorbar(
            right_x,
            right_y,
            yerr=right_e,
            color=color1,
            alpha=1,
            fmt="none",
            capsize=4,
        )

    if thr is not None and np.isfinite(thr):
        ax1.plot(
            [min(right_x) if right_x else 1e5, max(right_x) if right_x else 1e9],
            [thr, thr],
            linestyle="dashed",
            c="k",
            alpha=0.5,
            label="Positivity threshold",
        )
        ax0.plot([-1, 1], [thr, thr], linestyle="dashed", c="k", alpha=0.5)

    ax1.set_xlabel("S. pyogenes concentration in saliva [CFU/mL]", fontsize=14)
    f.suptitle(f"{selected_ds_raw} (combined N1/N2/N3)", fontsize=12, y=0.98)

    ax1.legend(facecolor="white", framealpha=0.9, loc="lower left")
    plt.subplots_adjust(wspace=0.05, hspace=0)

    buf = io.BytesIO()
    plt.tight_layout()
    f.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(f)
    buf.seek(0)
    return buf.getvalue()


def _final_graph_export_png_bytes(date: str, ds_raw: str) -> bytes:
    # Single replicate graph from exported CSV (used by /final_graph_selected.png).
    df = _load_export_csv_for_dataset(date, ds_raw)
    if df is None or df.empty:
        return _png_message(
            "Final graph (replicate): no exported CSV.\n Export CSV for dataset first."
        )

    loads: List[float] = []
    means: List[float] = []
    sems: List[float] = []
    labels: List[str] = []

    for row in df.index.tolist():
        r = _normalize_row_label(str(row))
        bl = _row_to_bacterial_load(r)
        if bl is None:
            continue

        try:
            mean_v = df.loc[r].get("row_ratio_mean", "")
            sem_v = df.loc[r].get("row_ratio_sem", "")
        except Exception:
            continue

        if mean_v in ("", None):
            continue
        try:
            mean_f = float(mean_v)
        except Exception:
            continue
        if not np.isfinite(mean_f):
            continue

        try:
            sem_f = float(sem_v) if str(sem_v).strip() != "" else 0.0
            if not np.isfinite(sem_f):
                sem_f = 0.0
        except Exception:
            sem_f = 0.0

        loads.append(float(bl))
        means.append(float(mean_f))
        sems.append(float(sem_f))
        labels.append(r)

    if not loads:
        return _png_message("Final graph (replicate): no plotted points (need row_ratio_mean).")

    order = np.argsort(np.array(loads, dtype=float)).tolist()
    loads = [loads[i] for i in order]
    means = [means[i] for i in order]
    sems = [sems[i] for i in order]
    labels = [labels[i] for i in order]

    thr = _compute_threshold_from_df(df, loads, means, sems, labels)

    left_x, left_y, left_e = [], [], []
    right_x, right_y, right_e = [], [], []

    for bl, m, e in zip(loads, means, sems):
        if float(bl) <= 0.0:
            left_x.append(float(bl))
            left_y.append(float(m))
            left_e.append(float(e))
        else:
            right_x.append(float(bl))
            right_y.append(float(m))
            right_e.append(float(e))

    y_all = np.array(means, dtype=float)
    y_min = float(np.nanmin(y_all)) if y_all.size else 0.99
    y_max = float(np.nanmax(y_all)) if y_all.size else 1.05
    if thr is not None and np.isfinite(thr):
        y_min = min(y_min, float(thr))
        y_max = max(y_max, float(thr))
    y_pad = (y_max - y_min) * 0.10 if (y_max > y_min) else 0.05
    y_lo = max(0.0, y_min - y_pad)
    y_hi = y_max + y_pad

    f, axes = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [1, 6]}, figsize=(9.6, 3.8)
    )
    ax0, ax1 = axes[0], axes[1]
    ax1.yaxis.tick_right()

    if right_x:
        ax1.set_xscale("log")

    ax0.set_ylim(y_lo, y_hi)
    ax1.set_ylim(y_lo, y_hi)

    ax0.set_xlim(-0.9, 1.0)
    ax0.set_xticks([0.0])
    ax0.set_xticklabels(["neg"])
    ax0.set_ylabel("Signal to baseline ratio", fontsize=12)

    ax0.yaxis.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax0.xaxis.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax1.grid(which="major", alpha=0.35, linewidth=0.8, zorder=0)
    ax1.grid(which="minor", alpha=0.22, linestyle=":", linewidth=0.6, zorder=0)
    ax1.minorticks_on()

    s = 45
    if left_x:
        ax0.scatter(
            left_x,
            left_y,
            facecolors="white",
            edgecolor="deepskyblue",
            linewidths=1,
            zorder=3,
            s=s,
        )
        ax0.errorbar(
            left_x,
            left_y,
            yerr=left_e,
            fmt="none",
            ecolor="deepskyblue",
            capsize=4,
            zorder=2,
        )

    if right_x:
        ax1.scatter(
            right_x,
            right_y,
            facecolors="white",
            edgecolor="deepskyblue",
            linewidths=1,
            zorder=3,
            s=s,
        )
        ax1.errorbar(
            right_x,
            right_y,
            yerr=right_e,
            fmt="none",
            ecolor="deepskyblue",
            capsize=4,
            zorder=2,
        )

    if thr is not None and np.isfinite(thr):
        ax1.plot(
            [min(right_x) if right_x else 1e5, max(right_x) if right_x else 1e9],
            [thr, thr],
            linestyle="dashed",
            c="k",
            alpha=0.5,
            label="Positivity threshold",
        )
        ax0.plot([-1, 1], [thr, thr], linestyle="dashed", c="k", alpha=0.5)

    ax1.set_xlabel("S. pyogenes concentration in saliva [CFU/mL]", fontsize=12)
    f.suptitle(f"{ds_raw}", fontsize=12, y=0.98)

    if right_x:
        xmin = min(right_x)
        xmax = max(right_x)
        if xmin > 0:
            ax1.set_xlim(max(1e-3, 0.5 * xmin), 2.0 * xmax)

    plt.subplots_adjust(wspace=0.05, hspace=0)

    buf = io.BytesIO()
    plt.tight_layout()
    f.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close(f)
    buf.seek(0)
    return buf.getvalue()


# =========================
# Image pipeline + normalization
# =========================
def _dtype_max_value(arr: np.ndarray) -> float:
    if arr is None:
        return 1.0
    if arr.dtype == np.uint8:
        return 255.0
    if arr.dtype == np.uint16:
        return 65535.0
    try:
        mx = float(np.nanmax(arr))
        return mx if mx > 0 else 1.0
    except Exception:
        return 1.0


def _to_gray_like_notebook(img: np.ndarray) -> np.ndarray:
    if img is None:
        raise ValueError("No image.")
    if img.ndim == 2:
        return img.astype(np.float32)
    return np.mean(img.astype(np.float32), axis=2)


def _normalize_gray_to_01(gray: np.ndarray) -> np.ndarray:
    g = np.asarray(gray, dtype=np.float32)
    if g.size == 0 or (not np.isfinite(g).any()):
        return np.zeros_like(g, dtype=np.float32)
    if float(np.nanmax(g)) > 1.5:
        denom = _dtype_max_value(gray)
        if denom <= 0:
            denom = float(np.nanmax(g)) if float(np.nanmax(g)) > 0 else 1.0
        g = g / float(denom)
    return np.clip(g, 0.0, 1.0).astype(np.float32)


def _autocontrast_u8(gray01: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.0) -> np.ndarray:
    g = np.asarray(gray01, dtype=np.float32)
    if g.size == 0:
        return np.zeros((1, 1), dtype=np.uint8)

    finite = g[np.isfinite(g)]
    if finite.size == 0:
        return np.zeros(g.shape, dtype=np.uint8)

    lo = float(np.percentile(finite, p_lo))
    hi = float(np.percentile(finite, p_hi))
    if (not np.isfinite(lo)) or (not np.isfinite(hi)) or hi <= lo:
        lo = float(np.min(finite))
        hi = float(np.max(finite))
        if hi <= lo:
            return (np.clip(g, 0.0, 1.0) * 255.0).astype(np.uint8)

    out = (g - lo) / (hi - lo)
    out = np.clip(out, 0.0, 1.0)
    return (out * 255.0).astype(np.uint8)


def _apply_grayscale_preserve_dtype(img: np.ndarray) -> np.ndarray:
    if img is None:
        raise ValueError("No image.")
    if img.ndim == 2:
        return img
    g = np.mean(img.astype(np.float32), axis=2)

    if img.dtype == np.uint8:
        return np.clip(g, 0, 255).astype(np.uint8)
    if img.dtype == np.uint16:
        return np.clip(g, 0, 65535).astype(np.uint16)

    return g.astype(np.float32)


def _load_image_by(dataset: str, index: int):
    files = DATASETS.get(dataset, [])
    if index < 0 or index >= len(files):
        return None
    path = files[index]["path"]
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return img


def _normalized_roi_from_current() -> np.ndarray:
    # IMPORTANT: this ROI is what drives ALL math. We normalize to 0..1, then invert in 0..1 so
    # dark lines become peaks (not minima), matching notebook-style "signal" direction.
    if CURRENT_IMAGE is None:
        raise ValueError("No current image.")

    gray = _to_gray_like_notebook(CURRENT_IMAGE)
    gray = cv2.resize(gray, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA).astype(
        np.float32
    )
    roi01 = _normalize_gray_to_01(gray)

    if ANALYSIS_INVERT_INTENSITY:
        roi01 = 1.0 - roi01

    return np.clip(roi01, 0.0, 1.0).astype(np.float32)


def _dataset_rows_from_files(ds_raw: str) -> List[str]:
    rows = [
        sanitize_name(row_name_from_filename(f["filename"]))
        for f in DATASETS.get(ds_raw, [])
    ]
    return sorted(set(rows), key=row_sort_key)


def _first_match(base_dir: Path, dataset: str, row: str) -> Optional[str]:
    folder = base_dir / dataset
    if not folder.exists():
        return None
    prefix = f"ROI_{row}_"
    for f in sorted(folder.glob(prefix + "*.tif*")):
        return str(f)
    return None


def _roi_to_demo_test_gray(roi: np.ndarray, half_band: Optional[int] = None) -> np.ndarray:
    # If half_band is None -> use ALL rows (fixes 'smaller window' look and matches notebook style).
    # If half_band is an int -> keep your older center-band behavior.
    if roi.ndim != 2:
        raise ValueError("Expected ROI to be 2D (H×W).")
    sig = roi.astype(np.float32)

    if half_band is None:
        return sig

    H = int(sig.shape[0])
    mid = H // 2
    a = max(0, mid - int(half_band))
    b = min(H, mid + int(half_band))
    return sig[a:b].astype(np.float32)


def _sbr_profile_from_roi(roi: np.ndarray, half_band: Optional[int] = None) -> np.ndarray:
    test = _roi_to_demo_test_gray(roi, half_band=half_band)
    if test.size == 0:
        return np.array([], dtype=np.float32)
    return np.mean(test, axis=0).astype(np.float32)


def _median_window(profile: np.ndarray, x: int, half_window: int = 5) -> float:
    lo = x - half_window
    hi = x + half_window
    if lo < 0 or hi >= profile.size:
        raise ValueError(
            f"Index {x} out of range for ±{half_window} window (n={profile.size})."
        )
    return float(np.median(profile[lo : hi + 1]))


def _peak_between(profile: np.ndarray, x1: int, x2: int) -> Tuple[int, float]:
    lo = int(min(x1, x2))
    hi = int(max(x1, x2))
    seg = profile[lo : hi + 1]
    if seg.size == 0:
        raise ValueError("Empty peak segment.")
    j = int(np.argmax(seg))
    peak_x = lo + j
    peak_val = float(seg[j])
    return int(peak_x), float(peak_val)


def _baseline_value(x_b: int, x_a: int, b: float, a: float) -> Tuple[float, float]:
    if x_a == x_b:
        raise ValueError("x_a equals x_b; cannot compute baseline.")
    x_half = (float(x_b) + float(x_a)) / 2.0
    baseline = float(b) + (
        (float(a) - float(b))
        * (x_half - float(x_b))
        / (float(x_a) - float(x_b))
    )
    return x_half, baseline


def _load_roi_from_path(path_str: str) -> Optional[np.ndarray]:
    if not path_str:
        return None
    p = Path(path_str)
    if not p.exists():
        return None
    try:
        arr = np.load(str(p))
        if arr.ndim != 2:
            return None
        return arr.astype(np.float32)
    except Exception:
        return None


def _arr_stats_01(arr: np.ndarray, tol: float = 1e-6) -> Dict[str, object]:
    a = np.asarray(arr)
    finite_mask = np.isfinite(a)
    if not finite_mask.any():
        return {
            "ok": False,
            "min": None,
            "max": None,
            "nan_count": int(a.size),
            "oor_count": int(a.size),
        }
    finite = a[finite_mask]
    mn = float(np.min(finite))
    mx = float(np.max(finite))
    oor = np.sum((finite < (0.0 - tol)) | (finite > (1.0 + tol)))
    nan_count = int(a.size - finite.size)
    ok = (oor == 0) and (nan_count == 0)
    return {
        "ok": bool(ok),
        "min": mn,
        "max": mx,
        "nan_count": nan_count,
        "oor_count": int(oor),
    }


# ----------------- Demo helpers -----------------
def snap_local(avg: np.ndarray, x_center: int, target_y: float, halfwin: int = 20) -> int:
    a = max(0, int(x_center) - int(halfwin))
    b = min(int(len(avg)) - 1, int(x_center) + int(halfwin))
    seg = avg[a : b + 1]
    rel = int(np.argmin(np.abs(seg - float(target_y))))
    return int(a + rel)


def baseline_y(x_left: int, y_left: float, x_right: int, y_right: float, x_mid: int) -> float:
    if int(x_right) == int(x_left):
        return float(y_left)
    t = (float(x_mid) - float(x_left)) / (float(x_right) - float(x_left))
    return float(y_left) + t * (float(y_right) - float(y_left))


def analyze_experiments(R_list: List[np.ndarray]) -> Dict[str, object]:
    if not R_list:
        return {
            "experiment_means": np.array([]),
            "experiment_SEMs": np.array([]),
            "experiment_ns": np.array([]),
            "R_best": float("nan"),
            "SE_best": float("nan"),
            "N_total": 0,
            "R_all": np.array([]),
        }

    exp_means = np.array([np.mean(R) for R in R_list], dtype=float)
    exp_sds = np.array([np.std(R, ddof=1) for R in R_list], dtype=float)
    exp_ns = np.array([int(R.size) for R in R_list], dtype=int)
    exp_sems = exp_sds / np.sqrt(exp_ns.astype(float))

    R_all = np.concatenate(R_list).astype(float)
    N = int(R_all.size)
    R_best = float(np.mean(R_all))
    s_pooled = float(np.std(R_all, ddof=1))
    SE_best = float(s_pooled / np.sqrt(float(N))) if N else float("nan")

    return {
        "experiment_means": exp_means,
        "experiment_SEMs": exp_sems,
        "experiment_ns": exp_ns,
        "R_best": R_best,
        "SE_best": SE_best,
        "N_total": N,
        "R_all": R_all,
    }


def _compute_demo_metrics_from_roi(
    roi: np.ndarray,
    x_blue: int,
    x_red: int,
    *,
    half_band: Optional[int] = None,
    snap_halfwin: int = 20,
    endpoint_median_halfwin: int = 5,
    endpoint_y_median_span: int = 10,
) -> Dict[str, object]:
    test = _roi_to_demo_test_gray(roi, half_band=half_band)
    if test.size == 0:
        raise ValueError("Empty ROI/test band.")

    row_averages = np.mean(test, axis=0).astype(np.float32)
    n = int(row_averages.size)
    if n < 20:
        raise ValueError("ROI too small for Demo analysis.")

    dn1 = float(
        np.median(
            row_averages[
                x_blue - endpoint_median_halfwin : x_blue + endpoint_median_halfwin + 1
            ]
        )
    )
    dn2 = float(
        np.median(
            row_averages[
                x_red - endpoint_median_halfwin : x_red + endpoint_median_halfwin + 1
            ]
        )
    )

    base_x_1 = int(snap_local(row_averages, x_blue, dn1, halfwin=snap_halfwin))
    base_x_2 = int(snap_local(row_averages, x_red, dn2, halfwin=snap_halfwin))
    if base_x_2 < base_x_1:
        base_x_1, base_x_2 = base_x_2, base_x_1

    peak_x = int(base_x_1 + int(np.argmax(row_averages[base_x_1 : base_x_2 + 1])))
    peak_y = float(row_averages[peak_x])

    y1_end = float(
        np.median(row_averages[base_x_1 : min(n, base_x_1 + endpoint_y_median_span)])
    )
    y2_end = float(
        np.median(row_averages[base_x_2 : min(n, base_x_2 + endpoint_y_median_span)])
    )
    baseline_at_peak = float(baseline_y(base_x_1, y1_end, base_x_2, y2_end, peak_x))

    sbr_profile = (
        float(peak_y / baseline_at_peak) if baseline_at_peak != 0 else float("inf")
    )

    peak_values = test[:, peak_x].astype(np.float32)
    y_left = test[:, base_x_1].astype(np.float32)
    y_right = test[:, base_x_2].astype(np.float32)
    slope = (y_right - y_left) / float(base_x_2 - base_x_1)
    baseline_values = y_left + slope * float(peak_x - base_x_1)

    row_ratios = (peak_values / baseline_values).astype(np.float32)

    H = int(row_ratios.size)
    row_mean = float(np.mean(row_ratios)) if H else float("nan")
    row_sd = float(np.std(row_ratios, ddof=1)) if H > 1 else float("nan")
    row_sem = float(row_sd / np.sqrt(H)) if H > 1 else float("nan")

    return {
        "base_x_1": int(base_x_1),
        "base_x_2": int(base_x_2),
        "peak_x": int(peak_x),
        "peak_y": float(peak_y),
        "baseline_at_peak": float(baseline_at_peak),
        "sbr_profile": float(sbr_profile),
        "row_ratios": row_ratios,
        "row_ratio_mean": row_mean,
        "row_ratio_sem": row_sem,
        "row_ratio_n": int(H),
    }


def _get_row_name_from_args() -> Tuple[Optional[str], Optional[str]]:
    if DF_GRAY is None or DF_GRAY.empty:
        return None, "No exported CSV/DF yet. Click 'Export CSV for Dataset' first."
    row_arg = (request.args.get("row", "") or "").strip()
    idx_arg = request.args.get("index", "0")
    if row_arg and row_arg in DF_GRAY.index:
        return str(row_arg), None
    try:
        idx = int(idx_arg)
    except ValueError:
        idx = 0
    if idx < 0 or idx >= len(DF_GRAY):
        return None, f"Row index out of range (0..{len(DF_GRAY) - 1})."
    return str(DF_GRAY.index[idx]), None


def _get_roi_for_row(row_name: str) -> Tuple[Optional[np.ndarray], Optional[str]]:
    roi_path = str(DF_GRAY.loc[row_name].get("roi_path", "") or "")
    roi = _load_roi_from_path(roi_path)
    if roi is None:
        return None, f"No ROI file found for row '{row_name}'. Save ROI first."
    return roi, None


In [7]:
# /content/uwlfa_gui_cells/cell7_routes.py

@app.route("/")
def home():
    html = TEMPLATE_HTML.replace("__PAYLOAD__", json.dumps(payload))
    return Response(html, mimetype="text/html")


@app.route("/export_info")
def export_info():
    full = (EXPORT_ROOT / DATE_FOLDER).as_posix()
    return jsonify({"root": str(EXPORT_ROOT), "date": DATE_FOLDER, "full": full})


@app.route("/datasets")
def datasets():
    return jsonify({"names": list(DATASETS.keys())})


@app.route("/row_name")
def row_name():
    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))
    files = DATASETS.get(ds, [])
    if 0 <= idx < len(files):
        return jsonify({"row": row_name_from_filename(files[idx]["filename"])})
    return jsonify({"row": ""})


@app.route("/images")
def images():
    ds = request.args.get("dataset", "")
    return jsonify({"files": DATASETS.get(ds, [])})


@app.route("/image")
def image():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    ds = request.args.get("dataset", "")
    idx = int(request.args.get("index", "0"))

    key = (ds, idx)
    if getattr(app, "_current_key", None) != key:
        img = _load_image_by(ds, idx)
        if img is None:
            img = np.zeros((40, 120, 3), np.uint8)
        ORIGINAL_IMAGE = img.copy()
        CURRENT_IMAGE = img.copy()
        app._current_key = key

    if CURRENT_IMAGE is None:
        return Response(_png_message("No image loaded."), mimetype="image/png")

    # DISPLAY uses DISPLAY_INVERT_INTENSITY (preview only),
    # analysis/saving uses ANALYSIS_INVERT_INTENSITY in _normalized_roi_from_current()
    if CURRENT_IMAGE.ndim == 2:
        gray = _to_gray_like_notebook(CURRENT_IMAGE)
        gray01 = _normalize_gray_to_01(gray)
        if DISPLAY_INVERT_INTENSITY:
            gray01 = 1.0 - gray01
        gray_u8 = _autocontrast_u8(gray01, p_lo=1.0, p_hi=99.0)
        rgb = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2RGB)
        ok, buf = cv2.imencode(".png", rgb)
        if not ok:
            return Response(_png_message("PNG encode failed."), mimetype="image/png")
        return Response(buf.tobytes(), mimetype="image/png")

    img = CURRENT_IMAGE
    img_f = img.astype(np.float32)
    if float(np.nanmax(img_f)) > 1.5:
        denom = _dtype_max_value(img)
        denom = denom if denom > 0 else float(np.nanmax(img_f)) if float(np.nanmax(img_f)) > 0 else 1.0
        img_f = img_f / float(denom)
    img_f = np.clip(img_f, 0.0, 1.0)
    if DISPLAY_INVERT_INTENSITY:
        img_f = 1.0 - img_f

    img_u8 = (img_f * 255.0).clip(0, 255).astype(np.uint8)
    rgb_u8 = cv2.cvtColor(img_u8, cv2.COLOR_BGR2RGB)

    ok, buf = cv2.imencode(".png", rgb_u8)
    if not ok:
        return Response(_png_message("PNG encode failed."), mimetype="image/png")
    return Response(buf.tobytes(), mimetype="image/png")


@app.route("/reset_view", methods=["POST"])
def reset_view():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if ORIGINAL_IMAGE is not None:
        CURRENT_IMAGE = ORIGINAL_IMAGE.copy()
        return jsonify({"ok": True, "msg": "View reset."})
    return jsonify({"ok": False, "msg": "No image to reset."})


@app.route("/rotate", methods=["POST"])
def rotate():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})
    CURRENT_IMAGE = cv2.rotate(CURRENT_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    ORIGINAL_IMAGE = cv2.rotate(ORIGINAL_IMAGE, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return jsonify({"ok": True, "msg": "Rotated 90° CCW (persistent)."})


@app.route("/apply_grayscale", methods=["POST"])
def apply_grayscale():
    global CURRENT_IMAGE
    if CURRENT_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})
    try:
        CURRENT_IMAGE = _apply_grayscale_preserve_dtype(CURRENT_IMAGE)
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Apply grayscale failed: {type(e).__name__}: {e}"})
    return jsonify({"ok": True, "msg": "Applied grayscale to current view (persistent until Reset View)."})


@app.route("/apply_crop_zoom", methods=["POST"])
def apply_crop_zoom():
    global CURRENT_IMAGE, ORIGINAL_IMAGE
    if CURRENT_IMAGE is None or ORIGINAL_IMAGE is None:
        return jsonify({"ok": False, "msg": "No image loaded."})

    d = request.get_json(force=True)
    nx = float(d.get("nx", 0.0))
    ny = float(d.get("ny", 0.0))
    nw = float(d.get("nw", 0.0))
    nh = float(d.get("nh", 0.0))
    tW = int(d.get("tw", ORIGINAL_IMAGE.shape[1]))
    tH = int(d.get("th", ORIGINAL_IMAGE.shape[0]))

    H, W = CURRENT_IMAGE.shape[:2]
    x1 = int(max(0, min(W - 1, round(nx * W))))
    y1 = int(max(0, min(H - 1, round(ny * H))))
    x2 = int(max(0, min(W, round((nx + nw) * W))))
    y2 = int(max(0, min(H, round((ny + nh) * H))))
    if x2 <= x1 or y2 <= y1:
        return jsonify({"ok": False, "msg": "Empty selection."})

    crop = CURRENT_IMAGE[y1:y2, x1:x2]
    upscale = (crop.shape[1] < tW) or (crop.shape[0] < tH)
    interp = cv2.INTER_CUBIC if upscale else cv2.INTER_AREA
    CURRENT_IMAGE = cv2.resize(crop, (tW, tH), interpolation=interp)
    return jsonify({"ok": True, "msg": f"Cropped [{x1}:{x2}]×[{y1}:{y2}] → {tW}×{tH}."})


@app.route("/save_roi", methods=["POST"])
def save_roi():
    data = request.get_json(force=True)
    ds_raw = data.get("dataset", "")
    idx = int(data.get("index", 0))

    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}"})
    files = DATASETS.get(ds_raw, [])
    if not (0 <= idx < len(files)):
        return jsonify({"ok": False, "msg": "Bad image index."})

    ds = sanitize_name(ds_raw)
    row = sanitize_name(row_name_from_filename(files[idx]["filename"]))
    row = _normalize_row_label(row)

    try:
        roi = _normalized_roi_from_current()
    except Exception as e:
        return jsonify({"ok": False, "msg": f"ROI not available: {e}"})

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_ds_dir = out_date_dir / ds
    out_ds_dir.mkdir(parents=True, exist_ok=True)

    key = (ds, row)
    ROI_SAVE_COUNT[key] = ROI_SAVE_COUNT.get(key, 0) + 1
    n = ROI_SAVE_COUNT[key]

    out_npy = out_ds_dir / f"ROI_{row}_{n}.npy"
    out_tif = out_ds_dir / f"ROI_{row}_{n}.tiff"

    np.save(str(out_npy), roi.astype(np.float32))
    cv2.imwrite(str(out_tif), (roi * 255.0).clip(0, 255).astype(np.uint8))

    ROI_LATEST_PATH[key] = out_npy
    ROI_LATEST_SHAPE[key] = (int(roi.shape[0]), int(roi.shape[1]))
    ROI_BY_DATASET_ROW[key] = roi

    return jsonify({"ok": True, "msg": f"Saved ROI → {out_npy.name} + {out_tif.name} (dataset={ds_raw}, row={row})"})


@app.route("/export_df", methods=["POST"])
def export_df():
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    data = request.get_json(force=True) if request.data else {}
    ds_raw = (data.get("dataset") or "").strip()
    if ds_raw not in DATASETS:
        return jsonify({"ok": False, "msg": f"Unknown dataset: {ds_raw}", "html": ""})

    ds = sanitize_name(ds_raw)
    rows = _dataset_rows_from_files(ds_raw)

    records = []
    for row in rows:
        key = (ds, row)
        roi_path = ROI_LATEST_PATH.get(key)
        shape = ROI_LATEST_SHAPE.get(key)

        records.append(
            {
                "row": row,
                "roi_path": str(roi_path) if roi_path else "",
                "shape": f"{shape[0]}x{shape[1]}" if shape else "0x0",
                "x_blue": "",
                "median_blue": "",
                "x_red": "",
                "median_red": "",
                "peak_x": "",
                "peak_value": "",
                "baseline_xhalf": "",
                "baseline_value": "",
                "base_x_1": "",
                "base_x_2": "",
                "peak_y": "",
                "baseline_at_peak": "",
                "sbr_profile": "",
                "row_ratio_mean": "",
                "row_ratio_sem": "",
                "row_ratio_n": "",
            }
        )

    df = pd.DataFrame.from_records(records).set_index("row")

    out_date_dir = EXPORT_ROOT / DATE_FOLDER
    out_date_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_date_dir / f"{ds}.csv"
    df.to_csv(out_csv)

    DF_GRAY = df.copy()
    DF_DATASET = ds
    DF_CSV_PATH = out_csv

    return jsonify({"ok": True, "msg": f"Wrote {out_csv} (rows={len(df)})", "html": ""})


@app.route("/load_df", methods=["POST"])
def load_df():
    """Load an existing exported CSV into DF_GRAY so the SBR/medians UI edits the right replicate.

    Payload (JSON):
      - dataset: dataset name (raw, as shown in dropdown) OR sanitized name
      - date: export folder (defaults to DATE_FOLDER)

    Notes:
      - This does NOT create a new CSV; it loads EXPORT_ROOT/date/<sanitize(dataset)>.csv.
      - Use /export_df first if the file does not exist yet.
    """
    global DF_GRAY, DF_DATASET, DF_CSV_PATH

    data = request.get_json(force=True) if request.data else {}
    ds_raw = (data.get("dataset") or "").strip()
    date = (data.get("date") or "").strip() or DATE_FOLDER

    if not ds_raw:
        return jsonify({"ok": False, "msg": "Load DF: missing dataset."})

    # Accept sanitized dataset names too
    if ds_raw not in DATASETS:
        rev = None
        for k in DATASETS.keys():
            if sanitize_name(k) == ds_raw:
                rev = k
                break
        if rev is None:
            return jsonify({"ok": False, "msg": f"Load DF: unknown dataset: {ds_raw}"})
        ds_raw = rev

    ds = sanitize_name(ds_raw)
    base = EXPORT_ROOT / date
    if not base.exists():
        return jsonify({"ok": False, "msg": f"Load DF: date folder not found: {base.as_posix()}"})

    csv_path = base / f"{ds}.csv"
    if not csv_path.exists():
        return jsonify(
            {
                "ok": False,
                "msg": (
                    "Load DF: CSV not found."
                    f"Expected: {csv_path.as_posix()}"
                    "Click 'Export CSV for Dataset' first (then save at least one ROI)."
                ),
            }
        )

    try:
        df = pd.read_csv(csv_path, index_col="row")
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Load DF: read failed: {type(e).__name__}: {e}"})

    DF_GRAY = df.copy()
    DF_DATASET = ds
    DF_CSV_PATH = csv_path

    return jsonify({"ok": True, "msg": f"Loaded {csv_path.name} (rows={len(df)})", "dataset": ds_raw, "date": date})


@app.route("/df_rows")
def df_rows():
    if DF_GRAY is None or not isinstance(DF_GRAY, pd.DataFrame) or DF_GRAY.empty:
        return jsonify({"ok": False, "rows": [], "msg": "No exported CSV/DF yet. Click 'Export CSV for Dataset' first."})
    return jsonify({"ok": True, "rows": [str(x) for x in DF_GRAY.index.tolist()], "dataset": DF_DATASET or ""})


@app.route("/exports")
def list_exports():
    dates = []
    if EXPORT_ROOT.exists():
        for p in sorted(EXPORT_ROOT.iterdir()):
            if p.is_dir():
                dates.append(p.name)
    return jsonify({"dates": dates})


@app.route("/analyze_grid")
def analyze_grid():
    """3-column ROI grid (N1/N2/N3).

    Query params:
      - date: export folder name
      - dataset: any dataset in replicate family
      - include_dip: 0/1 (default 0)
    """
    try:
        date = (request.args.get("date", "") or "").strip()
        ds_raw = (request.args.get("dataset", "") or "").strip()
        include_dip = (request.args.get("include_dip", "0") or "0").strip().lower() in {"1", "true", "yes", "y"}

        if not date:
            return Response(_png_message("Grid error: missing date."), mimetype="image/png")
        if ds_raw not in DATASETS:
            return Response(_png_message(f"Grid error: dataset not found:\n{ds_raw}"), mimetype="image/png")


        base = EXPORT_ROOT / date
        if not base.exists():
            return Response(_png_message(f"Grid error: date folder not found:\n{base.as_posix()}"), mimetype="image/png")


        triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=ds_raw)
        reps = ["N1", "N2", "N3"]

        if not any(triplet.get(r) for r in reps):
            return Response(_png_message("Grid error: could not resolve N1/N2/N3 datasets."), mimetype="image/png")

        row_set: set = set()
        for rep in reps:
            ds_rep_raw = triplet.get(rep)
            if not ds_rep_raw:
                continue
            for r in _dataset_rows_from_files(ds_rep_raw):
                rr = _normalize_row_label(r)
                if rr in {"cc", "k"}:
                    continue
                if (rr in {"dip", "dips"}) and (not include_dip):
                    continue
                row_set.add(rr)

        if not row_set:
            return Response(_png_message("Grid error: no rows found (after filtering cc/k)."), mimetype="image/png")

        rows = [r for r in GRAPH_ROW_ORDER if r in row_set]
        rows.extend(sorted([r for r in row_set if r not in rows]))

        imgs: Dict[Tuple[int, int], Optional[np.ndarray]] = {}
        found_any = False

        for j, rep in enumerate(reps):
            ds_rep_raw = triplet.get(rep)
            if not ds_rep_raw:
                for i in range(len(rows)):
                    imgs[(i, j)] = None
                continue

            ds_rep = sanitize_name(ds_rep_raw)
            for i, row in enumerate(rows):
                path = _first_match(base, ds_rep, row)
                if path:
                    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        imgs[(i, j)] = img.astype(np.float32) / 255.0
                        found_any = True
                        continue
                imgs[(i, j)] = None

        if not found_any:
            msg = "Grid error: no ROI TIFFs found for any of N1/N2/N3.\n\nResolved:\n"
            for rep in reps:
                msg += f"{rep}: {triplet.get(rep) or '(missing)'}\n"

            return Response(_png_message(msg), mimetype="image/png")

        nrows = len(rows)
        ncols = 3

        fig_w = 10.5
        fig_h = max(3.0, 0.65 * nrows)
        fig, axs = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
        axs = np.atleast_2d(axs)

        for j, rep in enumerate(reps):
            label = rep
            ds_label = triplet.get(rep) or ""
            if ds_label:
                label = f"{rep}\n{ds_label}"

            axs[0, j].set_title(label, fontsize=10)

        for i, row in enumerate(rows):
            for j in range(ncols):
                ax = axs[i, j]
                im = imgs.get((i, j))
                if im is not None:
                    ax.imshow(im, cmap="gray", aspect="auto")
                ax.set_axis_off()
            axs[i, 0].text(-0.03, 0.5, row, va="center", ha="right", transform=axs[i, 0].transAxes, fontsize=9)

        plt.suptitle(f"ROIs from {date} (N1/N2/N3)", y=0.995, fontsize=12)
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return Response(buf.getvalue(), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"Grid crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")



@app.route("/sbr_profile_json")
def sbr_profile_json():
    if DF_GRAY is None or not isinstance(DF_GRAY, pd.DataFrame) or DF_GRAY.empty:
        return jsonify({"ok": False, "msg": "No exported CSV/DF yet. Click 'Export CSV for Dataset' first."})

    row_name, err = _get_row_name_from_args()
    if err:
        return jsonify({"ok": False, "msg": err})

    roi, err = _get_roi_for_row(row_name)
    if err:
        return jsonify({"ok": False, "msg": err})

    # Use ALL rows by default (half_band=None)
    profile = _sbr_profile_from_roi(roi, half_band=None)
    return jsonify({"ok": True, "dataset": DF_DATASET or "", "row_name": row_name, "n": int(profile.size), "profile": profile.astype(float).tolist()})


@app.route("/update_medians", methods=["POST"])
def update_medians():
    global DF_GRAY, DF_CSV_PATH
    if DF_GRAY is None or DF_CSV_PATH is None:
        return jsonify({"ok": False, "msg": "No exported CSV yet. Click 'Export CSV for Dataset' first."})

    data = request.get_json(force=True)
    row_arg = (data.get("row") or "").strip()
    idx_arg = data.get("index", 0)

    if row_arg and row_arg in DF_GRAY.index:
        row_name = row_arg
    else:
        try:
            idx = int(idx_arg)
        except Exception:
            idx = 0
        if idx < 0 or idx >= len(DF_GRAY):
            return jsonify({"ok": False, "msg": f"Row index out of range (0..{len(DF_GRAY)-1})."})
        row_name = str(DF_GRAY.index[idx])

    try:
        x_blue = int(data.get("x_blue"))
        x_red = int(data.get("x_red"))
    except Exception:
        return jsonify({"ok": False, "msg": "Bad payload. Expected {row/index, x_blue, x_red}."})

    roi, err = _get_roi_for_row(row_name)
    if err:
        return jsonify({"ok": False, "msg": err})

    # Use ALL rows by default (half_band=None)
    profile = _sbr_profile_from_roi(roi, half_band=None)
    n = int(profile.size)

    if x_blue < 5 or x_blue > n - 6:
        return jsonify({"ok": False, "msg": f"Blue index invalid. Valid range is 5..{n-6}."})
    if x_red < 5 or x_red > n - 6:
        return jsonify({"ok": False, "msg": f"Red index invalid. Valid range is 5..{n-6}."})

    try:
        med_blue = float(np.median(profile[x_blue - 5 : x_blue + 6]))
        med_red = float(np.median(profile[x_red - 5 : x_red + 6]))

        base_x_1 = int(snap_local(profile, x_blue, med_blue, halfwin=20))
        base_x_2 = int(snap_local(profile, x_red, med_red, halfwin=20))
        if base_x_2 < base_x_1:
            base_x_1, base_x_2 = base_x_2, base_x_1

        peak_x = int(base_x_1 + int(np.argmax(profile[base_x_1 : base_x_2 + 1])))
        peak_value = float(profile[peak_x])

        y_left = float(np.median(profile[base_x_1 : min(n, base_x_1 + 10)]))
        y_right = float(np.median(profile[base_x_2 : min(n, base_x_2 + 10)]))
        baseline_value = float(baseline_y(base_x_1, y_left, base_x_2, y_right, peak_x))

        baseline_xhalf = float(peak_x)
    except Exception as e:
        return jsonify({"ok": False, "msg": f"Computation failed: {e}"})

    for col in (
        "x_blue",
        "median_blue",
        "x_red",
        "median_red",
        "peak_x",
        "peak_value",
        "baseline_xhalf",
        "baseline_value",
        "base_x_1",
        "base_x_2",
        "peak_y",
        "baseline_at_peak",
        "sbr_profile",
        "row_ratio_mean",
        "row_ratio_sem",
        "row_ratio_n",
    ):
        if col not in DF_GRAY.columns:
            DF_GRAY[col] = ""

    DF_GRAY.at[row_name, "x_blue"] = int(x_blue)
    DF_GRAY.at[row_name, "median_blue"] = float(med_blue)
    DF_GRAY.at[row_name, "x_red"] = int(x_red)
    DF_GRAY.at[row_name, "median_red"] = float(med_red)
    DF_GRAY.at[row_name, "peak_x"] = int(peak_x)
    DF_GRAY.at[row_name, "peak_value"] = float(peak_value)
    DF_GRAY.at[row_name, "baseline_xhalf"] = float(baseline_xhalf)
    DF_GRAY.at[row_name, "baseline_value"] = float(baseline_value)

    # Demo should match the same band choice (ALL rows)
    demo = _compute_demo_metrics_from_roi(roi, x_blue=x_blue, x_red=x_red, half_band=None)
    DF_GRAY.at[row_name, "base_x_1"] = int(demo["base_x_1"])
    DF_GRAY.at[row_name, "base_x_2"] = int(demo["base_x_2"])
    DF_GRAY.at[row_name, "peak_y"] = float(demo["peak_y"])
    DF_GRAY.at[row_name, "baseline_at_peak"] = float(demo["baseline_at_peak"])
    DF_GRAY.at[row_name, "sbr_profile"] = float(demo["sbr_profile"])
    DF_GRAY.at[row_name, "row_ratio_mean"] = float(demo["row_ratio_mean"])
    DF_GRAY.at[row_name, "row_ratio_sem"] = float(demo["row_ratio_sem"])
    DF_GRAY.at[row_name, "row_ratio_n"] = int(demo["row_ratio_n"])

    DF_GRAY.to_csv(DF_CSV_PATH)

    return jsonify(
        {
            "ok": True,
            "msg": (
                f"Updated {DF_CSV_PATH.name}: row={row_name}, "
                f"blue@{x_blue} med={med_blue:.6f}, "
                f"red@{x_red} med={med_red:.6f}, "
                f"peak@{peak_x} val={peak_value:.6f}, "
                f"baseline@peak={baseline_value:.6f} (x={baseline_xhalf:.0f})"
            ),
            "peak_x": int(peak_x),
            "peak_value": float(peak_value),
            "baseline_xhalf": float(baseline_xhalf),
            "baseline_value": float(baseline_value),
        }
    )


@app.route("/validate_normalization")
def validate_normalization():
    tol = float(request.args.get("tol", "1e-6") or "1e-6")
    max_rows = int(request.args.get("max_rows", "250") or "250")

    if DF_GRAY is None or not isinstance(DF_GRAY, pd.DataFrame) or DF_GRAY.empty:
        return jsonify({"ok": False, "msg": "Validate: no exported CSV/DF yet. Click 'Export CSV for Dataset' first.", "offenders": []})

    offenders: List[Dict[str, object]] = []
    checked = 0

    for row_name in DF_GRAY.index.tolist():
        if checked >= max_rows:
            break
        row_name_str = str(row_name)

        roi_path = str(DF_GRAY.loc[row_name_str].get("roi_path", "") or "")
        roi = _load_roi_from_path(roi_path)
        if roi is None:
            continue

        checked += 1

        s_roi = _arr_stats_01(roi, tol=tol)
        if not s_roi["ok"]:
            offenders.append({"row": row_name_str, "what": "roi", "stats": s_roi})

        # Use ALL rows
        prof = _sbr_profile_from_roi(roi, half_band=None)
        s_prof = _arr_stats_01(prof, tol=tol)
        if not s_prof["ok"]:
            offenders.append({"row": row_name_str, "what": "sbr_profile_array", "stats": s_prof})

        try:
            test = _roi_to_demo_test_gray(roi, half_band=None)
            s_test = _arr_stats_01(test, tol=tol)
            if not s_test["ok"]:
                offenders.append({"row": row_name_str, "what": "demo_test_band", "stats": s_test})
        except Exception as e:
            offenders.append({"row": row_name_str, "what": "demo_test_band_error", "err": f"{type(e).__name__}: {e}"})

        for col in ("median_blue", "median_red", "peak_value", "baseline_value"):
            if col not in DF_GRAY.columns:
                continue
            v = DF_GRAY.loc[row_name_str].get(col, "")
            if v in ("", None):
                continue
            try:
                fv = float(v)
            except Exception:
                continue
            if not (-tol <= fv <= 1.0 + tol):
                offenders.append({"row": row_name_str, "what": f"csv.{col}", "value": fv})

    msg = f"Validate 0–1: checked {checked} row(s) with ROI files; offenders={len(offenders)} (tol={tol:g})."
    return jsonify({"ok": len(offenders) == 0, "msg": msg, "checked": checked, "offenders": offenders})


@app.route("/demo_full_json")
def demo_full_json():
    if DF_GRAY is None or DF_GRAY.empty:
        return jsonify({"ok": False, "msg": "Export CSV first."})

    row_name, err = _get_row_name_from_args()
    if err:
        return jsonify({"ok": False, "msg": err})

    roi, err = _get_roi_for_row(row_name)
    if err:
        return jsonify({"ok": False, "msg": err})

    try:
        x_blue = int(request.args.get("x_blue", ""))
        x_red = int(request.args.get("x_red", ""))
    except Exception:
        return jsonify({"ok": False, "msg": "Missing x_blue/x_red."})

    test = _roi_to_demo_test_gray(roi, half_band=None)
    first_row = test[0].astype(float).tolist() if test.shape[0] else []
    row_averages = np.mean(test, axis=0).astype(float).tolist() if test.size else []

    demo = _compute_demo_metrics_from_roi(roi, x_blue=x_blue, x_red=x_red, half_band=None)
    R_exp1 = demo["row_ratios"].astype(float)
    results = analyze_experiments([R_exp1])

    out = {
        "ok": True,
        "dataset": DF_DATASET or "",
        "row_name": row_name,
        "test_shape": [int(test.shape[0]), int(test.shape[1])],
        "first_row_profile": first_row,
        "row_averages": row_averages,
        "demo": {
            "base_x_1": int(demo["base_x_1"]),
            "base_x_2": int(demo["base_x_2"]),
            "peak_x": int(demo["peak_x"]),
            "peak_y": float(demo["peak_y"]),
            "baseline_at_peak": float(demo["baseline_at_peak"]),
            "sbr_profile": float(demo["sbr_profile"]),
            "row_ratio_mean": float(demo["row_ratio_mean"]),
            "row_ratio_sem": float(demo["row_ratio_sem"]),
            "row_ratio_n": int(demo["row_ratio_n"]),
        },
        "results": {
            "experiment_means": results["experiment_means"].astype(float).tolist(),
            "experiment_SEMs": results["experiment_SEMs"].astype(float).tolist(),
            "experiment_ns": results["experiment_ns"].astype(int).tolist(),
            "R_best": float(results["R_best"]),
            "SE_best": float(results["SE_best"]),
            "N_total": int(results["N_total"]),
            "R_all": results["R_all"].astype(float).tolist(),
        },
    }
    return jsonify(out)


@app.route("/demo_strip.png")
def demo_strip_png():
    row_name, err = _get_row_name_from_args()
    if err:
        return Response(_png_message(err), mimetype="image/png")
    roi, err = _get_roi_for_row(row_name)
    if err:
        return Response(_png_message(err), mimetype="image/png")

    test = _roi_to_demo_test_gray(roi, half_band=None)

    # Display-only contrast stretch for readability (does NOT affect analysis).
    try:
        vmin = float(np.percentile(test, 2))
        vmax = float(np.percentile(test, 98))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin, vmax = 0.0, 1.0
    except Exception:
        vmin, vmax = 0.0, 1.0

    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    ax.imshow(test, cmap="gray", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_title(f"imshow(test) — {DF_DATASET or ''} — row {row_name} (auto-contrast)")
    ax.set_axis_off()

    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")



@app.route("/demo_first_row.png")
def demo_first_row_png():
    row_name, err = _get_row_name_from_args()
    if err:
        return Response(_png_message(err), mimetype="image/png")
    roi, err = _get_roi_for_row(row_name)
    if err:
        return Response(_png_message(err), mimetype="image/png")

    test = _roi_to_demo_test_gray(roi, half_band=None)
    if test.shape[0] == 0:
        return Response(_png_message("Empty test crop."), mimetype="image/png")

    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    ax.plot(test[0])
    ax.set_xlabel("Position")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.35)
    ax.set_title(f"Profile from first row — {DF_DATASET or ''} — row {row_name} (0–1)")

    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")


@app.route("/demo_rows_overlay.png")
def demo_rows_overlay_png():
    row_name, err = _get_row_name_from_args()
    if err:
        return Response(_png_message(err), mimetype="image/png")
    roi, err = _get_roi_for_row(row_name)
    if err:
        return Response(_png_message(err), mimetype="image/png")

    test = _roi_to_demo_test_gray(roi, half_band=None)

    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    for r in test:
        ax.plot(r)
    ax.set_xlabel("Position")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.35)
    ax.set_title(f"Profile from all rows — {DF_DATASET or ''} — row {row_name} (0–1)")

    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")


@app.route("/demo_average.png")
def demo_average_png():
    row_name, err = _get_row_name_from_args()
    if err:
        return Response(_png_message(err), mimetype="image/png")
    roi, err = _get_roi_for_row(row_name)
    if err:
        return Response(_png_message(err), mimetype="image/png")

    test = _roi_to_demo_test_gray(roi, half_band=None)
    avg = np.mean(test, axis=0) if test.size else np.array([])

    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    if avg.size:
        ax.plot(avg)

        # Auto-scale y-limits to the data (SBR-style: tight + small padding)
        y = np.asarray(avg, dtype=float)
        y = y[np.isfinite(y)]
        if y.size:
            y_min = float(y.min())
            y_max = float(y.max())
        else:
            y_min, y_max = 0.0, 1.0

        pad = (y_max - y_min) * 0.08
        if not np.isfinite(pad) or pad <= 0:
            pad = 0.01

        ax.set_ylim(y_min - pad, y_max + pad)

    ax.set_xlabel("Position")
    ax.grid(True, alpha=0.35)
    ax.set_title(f"AVERAGE Profile for all rows — {DF_DATASET or ''} — row {row_name} (auto-scale)")

    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")



@app.route("/final_graph.png")
def final_graph_png():
    try:
        return Response(_final_graph_png_bytes(), mimetype="image/png")
    except Exception as e:
        return Response(_png_message(f"Final graph crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")



@app.route("/final_graph_combined.png")
def final_graph_combined_png():
    """Graphing.ipynb-style combined curve across replicates.

    Usage:
      /final_graph_combined.png?date=2-1-2026&dataset=11-1-23_n=3

    Dataset may be any member of the replicate group; we resolve N1/N2/N3 automatically.
    """
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        ds_raw = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")

        if not date:
            return Response(_png_message("Combined graph error: missing date."), mimetype="image/png")
        if not ds_raw:
            return Response(_png_message("Combined graph error: missing dataset."), mimetype="image/png")

        if ds_raw not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == ds_raw:
                    rev = k
                    break
            if rev is None:
                return Response(_png_message(f"Combined graph error: unknown dataset:\n{ds_raw}"), mimetype="image/png")

            ds_raw = rev

        return Response(_final_graph_combined_png_bytes(date=date, selected_ds_raw=ds_raw), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"Combined graph crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")



@app.route("/final_graph_selected.png")
def final_graph_selected_png():
    """Final graph for a specific replicate (N1/N2/N3) using exported CSVs.

    Params:
      - date
      - dataset (any dataset in family)
      - mode: N1|N2|N3
    """
    try:
        date = (request.args.get("date", "") or "").strip() or DATE_FOLDER
        ds_raw = (request.args.get("dataset", "") or "").strip() or (DF_DATASET or "")
        mode = (request.args.get("mode", "") or "").strip().upper()

        if mode not in {"N1", "N2", "N3"}:
            return Response(_png_message("final_graph_selected.png: mode must be N1, N2, or N3."), mimetype="image/png")
        if not date or not ds_raw:
            return Response(_png_message("final_graph_selected.png: missing date/dataset."), mimetype="image/png")

        if ds_raw not in DATASETS:
            rev = None
            for k in DATASETS.keys():
                if sanitize_name(k) == ds_raw:
                    rev = k
                    break
            if rev is None:
                return Response(_png_message(f"Unknown dataset:\n{ds_raw}"), mimetype="image/png")

            ds_raw = rev

        triplet = _resolve_triplet_datasets(date=date, selected_ds_raw=ds_raw)
        ds_for_mode = triplet.get(mode)
        if not ds_for_mode:
            return Response(_png_message(f"Could not resolve {mode} dataset for:\n{ds_raw}"), mimetype="image/png")


        return Response(_final_graph_export_png_bytes(date=date, ds_raw=ds_for_mode), mimetype="image/png")

    except Exception as e:
        return Response(_png_message(f"final_graph_selected crashed:\n{type(e).__name__}: {e}"), mimetype="image/png")



@app.route("/experiment_add", methods=["POST"])
def experiment_add():
    if DF_GRAY is None:
        return jsonify({"ok": False, "msg": "Export CSV first."})

    data = request.get_json(force=True)
    row_name = (data.get("row") or "").strip()
    try:
        x_blue = int(data.get("x_blue"))
        x_red = int(data.get("x_red"))
    except Exception:
        return jsonify({"ok": False, "msg": "Missing x_blue/x_red."})

    if not row_name or row_name not in DF_GRAY.index:
        return jsonify({"ok": False, "msg": "Invalid row name."})

    roi, err = _get_roi_for_row(row_name)
    if err:
        return jsonify({"ok": False, "msg": err})

    demo = _compute_demo_metrics_from_roi(roi, x_blue=x_blue, x_red=x_red, half_band=None)
    R = demo["row_ratios"].astype(float)

    EXP_R_LIST.append(np.array(R, dtype=float))
    EXP_LABELS.append(str(row_name))

    res = analyze_experiments(EXP_R_LIST)
    return jsonify(
        {
            "ok": True,
            "msg": f"Added experiment: {row_name} (rows={len(R)}). Total experiments: {len(EXP_R_LIST)}",
            "labels": EXP_LABELS,
            "R_best": float(res["R_best"]),
            "SE_best": float(res["SE_best"]),
            "N_total": int(res["N_total"]),
        }
    )


@app.route("/experiment_clear", methods=["POST"])
def experiment_clear():
    EXP_R_LIST.clear()
    EXP_LABELS.clear()
    return jsonify({"ok": True, "msg": "Cleared experiment list."})


@app.route("/experiment_plot.png")
def experiment_plot_png():
    if not EXP_R_LIST:
        return Response(_png_message("No experiments yet.\nClick 'Add as Experiment'."), mimetype="image/png")

    res = analyze_experiments(EXP_R_LIST)
    means = np.array([np.mean(R) for R in EXP_R_LIST], dtype=float)
    sems = np.array([np.std(R, ddof=1) / np.sqrt(R.size) for R in EXP_R_LIST], dtype=float)
    labels = list(EXP_LABELS)
    pooled = float(res["R_best"])
    pooled_sem = float(res["SE_best"])

    fig, ax = plt.subplots(figsize=(9.0, 3.8))
    x = np.arange(len(means))
    ax.bar(x, means)
    ax.errorbar(x, means, yerr=sems, fmt="none", capsize=4)
    ax.axhline(pooled, lw=2, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Mean row-level ratio")
    ax.set_title(f"Experiment summary (pooled={pooled:.4f} ± {pooled_sem:.4f})")
    ax.grid(True, axis="y", alpha=0.25)

    buf = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Response(buf.getvalue(), mimetype="image/png")


@app.route("/payload.json")
def payload_json():
    return jsonify(payload)

In [8]:
# /content/uwlfa_gui_cells/cell8_run_server.py

from werkzeug.serving import make_server  # noqa


class ServerThread(threading.Thread):
    def __init__(self, flask_app: Flask):
        super().__init__(daemon=True)
        self.srv = make_server("127.0.0.1", 0, flask_app)
        self.port = self.srv.server_port

    def run(self):
        self.srv.serve_forever()


server = ServerThread(app)
server.start()

try:
    from google.colab import output as colab_output  # type: ignore

    proxied = colab_output.eval_js(f"google.colab.kernel.proxyPort({server.port})")
    display(
        HTML(
            f'<a href="{proxied}" target="_blank" '
            f'style="font-weight:700;background:#2a6de0;color:#fff;'
            f'padding:10px 14px;border-radius:8px;text-decoration:none">Open UW-LFA GUI in a new tab</a>'
        )
    )
except Exception:
    print(f"Open: http://127.0.0.1:{server.port}")

print(f"Repo: {REPO_DIR}\nDatabase: {DB_DIR}\n")
print("Loaded folders:")
for k in DATASETS:
    print(f" - {k}: {[d['filename'] for d in DATASETS[k]]}")
print(f"\nSaving to: {(EXPORT_ROOT / DATE_FOLDER).as_posix()}")


Repo: /content/_uwlfa_tmp/UW-LFA-Analysis-main
Database: /content/_uwlfa_tmp/UW-LFA-Analysis-main/100microliters/Database

Loaded folders:
 - 11-1-23_n=3: ['1.5e5_n=3.jpg', '1e5_n=3.jpg', '1e7_n=3.jpg', '5e5_n=3.jpg', '5e6_n=3.jpg', 'CC_1e6_n=3.jpg', 'K_1e6_n=3.jpg', 'dip_n_3.jpg', 'neg_n_3.jpg', 'pos_n_3.jpg']
 - 8-3-23_n=1: ['1.5e5_n=1.jpg', '1e5_n=1.jpg', '1e6_n=1.jpg', '1e7_n=1.jpg', '5e5_n=1.jpg', '5e6_n=1.jpg', 'dip_n_1.jpg', 'neg_ctrl_n_1.jpg', 'pos_ctrl_n_1.jpg']
 - 9-6-23_n=2: ['1.5e5_n=2.jpg', '1e5_n=2.jpg', '1e6_n=2.jpg', '1e7_n=2.jpg', '5e5_n=2.jpg', '5e6_n=2.jpg', 'dips_n_2.jpg', 'neg_ctrls_n_2.jpg', 'pos_ctrl_n_2.jpg']
 - LossLessFormat_N1: ['1.5e5_n=1.tiff', '1e5_n=1.tiff', '1e6_n=1.tiff', '1e7_n=1.tiff', '5e5_n=1.tiff', '5e6_n=1.tiff', 'neg_ctrl_n_1.tiff', 'pos_ctrl_n_1.tiff']
 - LossLessFormat_N2: ['1.5e5_n=2.tiff', '1e5_n=2.tiff', '1e6_n=2.tiff', '1e7_n=2.tiff', '5e5_n=2.tiff', '5e6_n=2.tiff', 'neg_ctrls_n_2.tiff', 'pos_ctrl_n_2.tiff']
 - LossLessFormat_N3: ['1.5e5_n=